# Mission 3 ver24 — 의료 사전학습 모델과 일반 한국어 모델 비교

Colab GPU 런타임에서 **런타임 → 모두 실행**을 누르세요.
Drive 연결 시 입력 ZIP이 있는 **공용 계정**을 선택하세요.
필요한 업로드 파일은 이 노트북과 기존 `mission3_ver5_inputs.zip`입니다.
공개 사전학습 모델은 실행 중 내려받습니다. 이전 ver19~23 모델 파일은 필요하지 않습니다.
마지막 학습 셀만 실행하지 마세요. 준비 셀이 실패하거나 런타임을 초기화했다면 처음부터 실행하세요.

이번에는 **한국어·영어 의료 문장으로 사전학습된 bi-KM-BERT**와
**일반 한국어 KLUE-RoBERTa-base**를 같은 데이터 분할·손실함수·예산 조건에서 비교합니다.
원본 Training 데이터와 9개 증상 정답을 유지하며, 의심스러운 정답도 자동 삭제·수정하지 않습니다.
입력은 대화의 `utterances[].text`만 사용하고 각 대화의 앞부분을 최대 512토큰으로 처리합니다.
질문·부정·시간 표현을 별도 정답으로 만들지 않습니다. 외부 학습 데이터를 추가하지 않습니다.
의료 지식이 실제 오심 분류에 도움이 되는지는 이번 결과로 확인해야 합니다.
이 실험은 친구의 11개 모델 앙상블을 재현하거나 직접 비교하는 실행이 아닙니다.

**180분 예산에는 설치·모델 다운로드·데이터 준비·학습·최종 평가·저장이 모두 포함됩니다.**
Drive 로그인 대기 시간은 제외합니다. 두 모델의 짧은 속도 측정 후 공통 epoch 수를 정합니다.
전체 Training 재학습까지 끝낼 시간이 부족하면 미완료로 종료하며 완료 결과로 표시하지 않습니다.
느린 GPU나 네트워크에서는 모든 단계의 완료를 보장할 수 없습니다.
결과 표시 후와 오류·시간 제한 시 GPU 런타임 반환을 요청하며 결과는 Drive에 저장합니다.
F1 상승 및 실제 Colab 학습 속도는 아직 측정하지 않았습니다.


## 비교 조건과 모델 선택

| 후보 | 공개 사전학습 모델 | 동일하게 적용하는 분류 학습 |
|---|---|---|
| 일반 한국어 대조군 | `klue/roberta-base` | mean pooling + dropout + 9개 증상 판단층 |
| 의료 사전학습 후보 | `silber2258/bi-KM-BERT` | 동일한 분류 구조와 학습 설정 |

손실은 `BCEWithLogitsLoss`이며 각 증상의 양성 가중치는
`clip(sqrt(음성 수 / 양성 수), 1, 5)`로 학습에 쓰는 데이터에서만 계산합니다.
샘플 삭제·재라벨링·학습 정답 마스킹·증상별 손실함수 변경은 적용하지 않습니다.
두 모델은 같은 seed 42의 Training 내부 분할에서 학습하고 평가합니다.
정확히 같은 원문 대화는 한쪽 분할에만 들어갑니다. 공식 Validation은 분할·후보 선택에 사용하지 않습니다.
의료 모델의 tokenizer는 의료 모델과 함께 공개된 tokenizer를 사용합니다.
두 모델 모두 최대 512토큰, 학습률 0.00002, batch 8 × gradient accumulation 2를 사용합니다.
속도 측정 후 가능한 공통 학습 횟수를 최대 3 epoch 안에서 정하고, 각 후보의 내부 평가가 가장 좋은 epoch 수·예측·지표를 저장합니다.
비교용 fit 가중치는 보관하지 않고, 최종 전체 Training 재학습 가중치만 저장해 Drive 저장량을 줄입니다.
학습률 스케줄은 fit과 전체 Training 재학습 모두 이 공통 epoch 수를 기준으로 하며, 선택 epoch가 더 적어도 압축하지 않습니다.
같은 512토큰이어도 두 tokenizer가 포함하는 실제 원문 길이는 다를 수 있습니다.

**의료 후보의 Macro F1이 대조군보다 높고 오심 F1도 낮아지지 않을 때만 의료 후보를 선택합니다.**
그 외에는 이 실행에서 학습한 일반 한국어 대조군을 선택합니다.
이는 기존 최고 성능 앙상블보다 좋은 모델이라는 뜻이 아닙니다.
선택한 모델 종류와 epoch 수로 **전체 Training을 사전학습 가중치에서 새로 학습**한 뒤
최종 모델 하나를 저장하고 공식 Validation을 한 번 평가합니다.
예측은 모든 증상에 동일한 `logit >= 0` (sigmoid 확률 0.5) 규칙을 고정 적용합니다.
threshold 탐색·클래스별 기준 변경·사후 점수 보정·앙상블 비율 탐색을 하지 않습니다.
반복 사용한 Training holdout 성능은 탐색 결과이며 독립된 새 데이터의 성능을 보장하지 않습니다.


In [ ]:
# @title 실행 설정
SETTINGS = {'version': 'ver24', 'input_zip': '/content/drive/MyDrive/mission3_ver5_inputs.zip', 'output_dir': '/content/drive/MyDrive/DCC_AIdeas/results/mission3/ver24', 'work_dir': '/content/mission3_ver24', 'train_dir': '', 'validation_dir': '', 'run_validation': True, 'seed': 42, 'holdout_size': 0.2, 'batch_size': 8, 'max_length': 512, 'gradient_accumulation': 2, 'max_epochs': 3, 'learning_rate': 2e-05, 'weight_decay': 0.01, 'dropout': 0.1, 'warmup_ratio': 0.1, 'pilot_optimizer_steps': 10, 'allow_cpu': False, 'time_budget_minutes': 180, 'finalization_reserve_seconds': 1200, 'budget_safety_factor': 1.35, 'log_every': 100}
AUTO_RELEASE_GPU = True # @param {type:"boolean"}
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
# Each rerun starts a new budget and consumes additional Colab resources.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 3시간 작업 예산 시작 (이후 패키지 설치 시간도 포함)
import time
import threading
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
if not isinstance(globals().get("SETTINGS"), dict) or "AUTO_RELEASE_GPU" not in globals():
    raise RuntimeError("실행 설정이 없습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
BUDGET_STARTED_MONOTONIC = time.monotonic()
DEADLINE_MONOTONIC = BUDGET_STARTED_MONOTONIC + float(SETTINGS["time_budget_minutes"]) * 60
SETTINGS["deadline_monotonic"] = DEADLINE_MONOTONIC
previous_timer = globals().get("RELEASE_TIMER")
if previous_timer is not None:
    previous_timer.cancel()
RELEASE_TIMER = None
RELEASE_LOCK = threading.Lock()
RELEASE_REQUESTED = False

def release_gpu():
    global RELEASE_REQUESTED
    if not AUTO_RELEASE_GPU:
        return
    with RELEASE_LOCK:
        if RELEASE_REQUESTED:
            return
        RELEASE_REQUESTED = True
        if RELEASE_TIMER is not None:
            RELEASE_TIMER.cancel()
    try:
        from google.colab import runtime as colab_runtime
        print("Colab GPU 런타임 반환을 요청합니다. 저장된 결과는 Drive에서 확인하세요.", flush=True)
        colab_runtime.unassign()
    except Exception as error:
        print("자동 반환 요청 실패:", type(error).__name__,
              "· Colab 메뉴 '런타임 → 연결 해제 및 런타임 삭제'를 직접 실행하세요.", flush=True)

if AUTO_RELEASE_GPU:
    RELEASE_TIMER = threading.Timer(max(0, DEADLINE_MONOTONIC - time.monotonic()), release_gpu)
    RELEASE_TIMER.daemon = True
    RELEASE_TIMER.start()
print("작업 시간 예산:", SETTINGS["time_budget_minutes"], "분")


In [ ]:
# @title 실행 코드 준비 (코드 파일 추가 업로드 불필요)
import importlib.util
import sys
import tempfile
from pathlib import Path
RUNTIME_SOURCES = {'__init__.py': '"""Matched medical versus general Korean encoder experiment."""\n', 'data.py': '"""Text-only Mission 3 data, grouped Training holdout, and discrete predictions."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport re\nimport tempfile\nimport unicodedata\n\nimport numpy as np\nfrom sklearn.metrics import f1_score, precision_recall_fscore_support\n\nTARGET_SYMPTOMS = (\n    "고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란",\n)\nLEXICAL_PATTERNS = {\n    "nausea": r"오심|메스꺼|메스껍|메슥|매스꺼|매스껍|매슥|울렁|구역|미식|느글|토할",\n    "question": r"\\?|나요|까요|습니까|신가|세요",\n    "negation": r"아니|없|않|못|안\\s|아닌",\n}\nLEXICAL_COUNTS_NOTE = (\n    "Regex matches are review candidates only: questions and negation may appear elsewhere "\n    "in the dialogue. These flags do not establish annotation errors or modify labels/predictions."\n)\n\n\ndef dialogue_text(annotation: dict) -> str:\n    """Read only utterance text, keeping turn boundaries and chronological order."""\n    key = "utterances" if "utterances" in annotation else "utterences"\n    utterances = annotation.get(key)\n    if not isinstance(utterances, list):\n        raise ValueError("utterances/utterences must be a list")\n    texts = []\n    for utterance in utterances:\n        if not isinstance(utterance, dict) or not isinstance(utterance.get("text"), str):\n            raise ValueError("Each utterance must be an object with string text")\n        text = " ".join(unicodedata.normalize("NFC", utterance["text"]).split())\n        if text:\n            texts.append(text)\n    return "\\n".join(texts)\n\n\ndef target_vector(annotation: dict) -> list[int]:\n    if "symptom" not in annotation:\n        raise ValueError("Training/evaluation JSON has no symptom target")\n    raw = annotation["symptom"]\n    if raw is None:\n        raw = []\n    if isinstance(raw, str):\n        raw = [raw]\n    if not isinstance(raw, list) or any(not isinstance(item, str) for item in raw):\n        raise ValueError("symptom must be a string, a list of strings, or null")\n    found = {item.strip() for item in raw}\n    return [int(name in found) for name in TARGET_SYMPTOMS]\n\n\ndef text_group(text: str) -> str:\n    normalized = " ".join(unicodedata.normalize("NFC", text).split())\n    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()\n\n\ndef load_dataset(label_dir, include_targets: bool = False):\n    label_dir = Path(label_dir)\n    if not label_dir.is_dir():\n        raise ValueError(f"Label directory does not exist: {label_dir}")\n    paths = sorted(path for path in label_dir.rglob("*.json") if path.is_file())\n    if not paths:\n        raise ValueError(f"No label JSON files: {label_dir}")\n    if len({path.name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate JSON filenames make submission identifiers ambiguous")\n\n    def read(path):\n        with path.open(encoding="utf-8-sig") as handle:\n            annotation = json.load(handle)\n        if not isinstance(annotation, dict):\n            raise ValueError(f"Expected a JSON object: {path}")\n        # Inference never inspects symptom, even when a supplied value is malformed.\n        return dialogue_text(annotation), target_vector(annotation) if include_targets else None\n\n    with ThreadPoolExecutor(max_workers=8) as executor:\n        rows = list(executor.map(read, paths))\n    texts = [row[0] for row in rows]\n    targets = np.asarray([row[1] for row in rows], dtype=np.int8) if include_targets else None\n    return paths, texts, targets\n\n\ndef _binary(values, rows=None, name="predictions"):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and values.shape[0] != rows)\n            or not np.isin(values, [0, 1]).all()):\n        raise ValueError(f"{name} must have shape (samples, 9) and contain only discrete 0/1 values")\n    return values.astype(np.int8)\n\n\ndef _greedy_groups(features, sizes, holdout_size, seed):\n    """Dependency-free iterative stratification of indivisible transcript groups.\n\n    This balances label presence and absence, then row counts. Group constraints\n    mean neither this fallback nor iterstrat can guarantee exact proportions.\n    """\n    rng = np.random.default_rng(seed)\n    active = np.ones(len(features), dtype=bool)\n    assignment = np.full(len(features), -1, dtype=np.int8)\n    remaining = features.sum(axis=0).astype(float)\n    proportions = np.asarray([1.0 - holdout_size, holdout_size])\n    desired_labels = proportions[:, None] * remaining[None, :]\n    desired_rows = proportions * sizes.sum()\n    coverage = np.zeros((2, features.shape[1]), dtype=bool)\n    while active.any():\n        available = np.flatnonzero(remaining > 0)\n        if not len(available):\n            candidates = np.flatnonzero(active)\n            rng.shuffle(candidates)\n            label = None\n        else:\n            rare = available[remaining[available] == remaining[available].min()]\n            label = int(rng.choice(rare))\n            candidates = np.flatnonzero(active & features[:, label].astype(bool))\n            rng.shuffle(candidates)\n        for group in candidates:\n            deficits = desired_rows if label is None else desired_labels[:, label]\n            sides = np.flatnonzero(deficits == deficits.max())\n            if len(sides) > 1:\n                sides = sides[desired_rows[sides] == desired_rows[sides].max()]\n            present = np.flatnonzero(features[group])\n            final_occurrences = present[remaining[present] == 1]\n            required = np.unique(np.where(~coverage[:, final_occurrences])[0])\n            if len(required) == 1:\n                # Reserve the last group for a side still missing this target.\n                sides = required\n            side = int(rng.choice(sides))\n            assignment[group] = side\n            desired_rows[side] -= sizes[group]\n            desired_labels[side] -= features[group]\n            remaining -= features[group]\n            coverage[side] |= features[group].astype(bool)\n            active[group] = False\n    return np.flatnonzero(assignment == 0), np.flatnonzero(assignment == 1)\n\n\ndef make_split(texts, targets, holdout_size: float = .2, seed: int = 42):\n    """Split Training only; normalized duplicate transcripts cannot cross sides."""\n    if not 0 < holdout_size < .5:\n        raise ValueError("holdout_size must be in (0, .5)")\n    targets = _binary(targets, len(texts), "targets")\n    if any(not isinstance(text, str) for text in texts):\n        raise TypeError("texts must contain strings only")\n    groups = np.asarray([text_group(text) for text in texts])\n    unique, inverse, sizes = np.unique(groups, return_inverse=True, return_counts=True)\n    presence = np.zeros((len(unique), len(TARGET_SYMPTOMS)), dtype=np.int8)\n    absence = np.zeros_like(presence)\n    np.maximum.at(presence, inverse, targets)\n    np.maximum.at(absence, inverse, 1 - targets)\n    features = np.concatenate([presence, absence], axis=1)\n    if len(unique) < 2 or np.any(features.sum(axis=0) < 2):\n        raise ValueError("Each symptom needs positive and negative examples in at least two text groups")\n    try:\n        from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit\n    except ImportError:\n        MultilabelStratifiedShuffleSplit = None\n    best = None\n    # Retry only group allocation, never inspecting official Validation.\n    for attempt in range(8):\n        current_seed = seed + attempt\n        if MultilabelStratifiedShuffleSplit is None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n        else:\n            splitter = MultilabelStratifiedShuffleSplit(\n                n_splits=1, test_size=holdout_size, random_state=current_seed)\n            fit_groups, holdout_groups = next(splitter.split(np.arange(len(unique)), features))\n        fit = np.flatnonzero(np.isin(inverse, fit_groups))\n        holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n        valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                    and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if not valid and MultilabelStratifiedShuffleSplit is not None:\n            fit_groups, holdout_groups = _greedy_groups(features, sizes, holdout_size, current_seed)\n            fit = np.flatnonzero(np.isin(inverse, fit_groups))\n            holdout = np.flatnonzero(np.isin(inverse, holdout_groups))\n            valid = all(len(ids) and np.all(targets[ids].sum(axis=0) > 0)\n                        and np.all(targets[ids].sum(axis=0) < len(ids)) for ids in (fit, holdout))\n        if valid:\n            ratio = targets[holdout].sum(axis=0) / targets.sum(axis=0)\n            discrepancy = float(np.abs(ratio - holdout_size).mean()\n                                + abs(len(holdout) / len(targets) - holdout_size))\n            if best is None or discrepancy < best[0]:\n                best = (discrepancy, fit, holdout)\n    if best is None:\n        raise ValueError("Cannot split text groups with positive and negative examples for every symptom")\n    return best[1], best[2]\n\n\ndef _atomic_text(path, write, encoding="utf-8-sig"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding=encoding, newline="",\n                                         dir=path.parent, prefix=".write_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            write(handle)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef write_json(path, value):\n    _atomic_text(path, lambda handle: json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False),\n                 encoding="utf-8")\n\n\ndef write_predictions(paths, predictions, output):\n    """Write already discrete model decisions; this API has no score or threshold."""\n    predictions = _binary(predictions, len(paths))\n    if len({Path(path).name.casefold() for path in paths}) != len(paths):\n        raise ValueError("Duplicate prediction identifiers")\n\n    def write(handle):\n        writer = csv.writer(handle)\n        writer.writerow(["label file name", "symptom"])\n        for path, flags in zip(paths, predictions):\n            writer.writerow([Path(path).name, str([name for name, flag in zip(TARGET_SYMPTOMS, flags) if flag])])\n    _atomic_text(output, write)\n    return predictions\n\n\ndef score_predictions(targets, predictions):\n    targets = _binary(targets, name="targets")\n    predictions = _binary(predictions, len(targets))\n    if not len(targets):\n        raise ValueError("Cannot score an empty dataset")\n    precision, recall, f1, support = precision_recall_fscore_support(targets, predictions, zero_division=0)\n    return {\n        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),\n        "micro_f1": float(f1_score(targets, predictions, average="micro", zero_division=0)),\n        "samples": int(len(targets)),\n        "per_symptom": {name: {"f1": float(f1[i]), "precision": float(precision[i]),\n                                "recall": float(recall[i]), "support": int(support[i]),\n                                "false_positive": int(((targets[:, i] == 0) & (predictions[:, i] == 1)).sum()),\n                                "false_negative": int(((targets[:, i] == 1) & (predictions[:, i] == 0)).sum())}\n                        for i, name in enumerate(TARGET_SYMPTOMS)},\n    }\n\n\ndef training_diagnostics(texts, targets):\n    """Return aggregate counts only, including suspected nausea annotation ambiguity."""\n    targets = _binary(targets, len(texts), "targets")\n    groups = {}\n    for index, text in enumerate(texts):\n        groups.setdefault(text_group(text), []).append(index)\n    duplicates = [indices for indices in groups.values() if len(indices) > 1]\n    conflicts = [indices for indices in duplicates if len(np.unique(targets[indices], axis=0)) > 1]\n    nausea = targets[:, TARGET_SYMPTOMS.index("오심")].astype(bool)\n    vomiting = targets[:, TARGET_SYMPTOMS.index("구토")].astype(bool)\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    lexical = {key: int(flags.sum()) for key, flags in signals.items()}\n    for key in ("question", "negation"):\n        cooccurrence = signals["nausea"] & signals[key]\n        lexical[f"nausea_and_{key}"] = int(cooccurrence.sum())\n        lexical[f"nausea_and_{key}_label_positive"] = int((cooccurrence & nausea).sum())\n        lexical[f"nausea_and_{key}_label_negative"] = int((cooccurrence & ~nausea).sum())\n    lexical["nausea_mention_label_positive"] = int((signals["nausea"] & nausea).sum())\n    lexical["nausea_mention_label_negative"] = int((signals["nausea"] & ~nausea).sum())\n    return {\n        "samples": len(texts), "all_zero_samples": int((targets.sum(axis=1) == 0).sum()),\n        "empty_text_samples": sum(not text.strip() for text in texts),\n        "positive_counts": {name: int(targets[:, i].sum()) for i, name in enumerate(TARGET_SYMPTOMS)},\n        "text_groups": len(groups), "duplicate_groups": len(duplicates),\n        "duplicate_extra_samples": sum(len(indices) - 1 for indices in duplicates),\n        "conflicting_label_groups": len(conflicts),\n        "conflicting_label_samples": sum(len(indices) for indices in conflicts),\n        "nausea_vomiting_contingency": {\n            "neither": int((~nausea & ~vomiting).sum()), "nausea_only": int((nausea & ~vomiting).sum()),\n            "vomiting_only": int((~nausea & vomiting).sum()), "both": int((nausea & vomiting).sum())},\n        "lexical_cooccurrence_counts": lexical,\n        "lexical_patterns": LEXICAL_PATTERNS,\n        "lexical_counts_note": LEXICAL_COUNTS_NOTE,\n    }\n\n\ndef export_errors(paths, texts, targets, predictions, output_dir, max_per_label: int = 20):\n    """Save full text locally for manual review; never print sensitive dialogue."""\n    targets = _binary(targets, len(texts), "targets")\n    predictions = _binary(predictions, len(texts))\n    if len(paths) != len(texts) or type(max_per_label) is not int or max_per_label < 1:\n        raise ValueError("paths/texts must align and max_per_label must be a positive integer")\n    output_dir = Path(output_dir)\n    outputs = []\n    signals = {key: np.asarray([bool(re.search(pattern, text)) for text in texts])\n               for key, pattern in LEXICAL_PATTERNS.items()}\n    header = ["label file name", "symptom", "error_type", "true_symptoms", "predicted_symptoms", "text",\n              "review_category", "review_note", "has_nausea_expression", "has_question_expression",\n              "has_negation_expression"]\n\n    def review_row(index, symptom, category):\n        actual = [name for name, flag in zip(TARGET_SYMPTOMS, targets[index]) if flag]\n        predicted = [name for name, flag in zip(TARGET_SYMPTOMS, predictions[index]) if flag]\n        return [Path(paths[index]).name, symptom, category, str(actual), str(predicted), texts[index], "", "",\n                int(signals["nausea"][index]), int(signals["question"][index]), int(signals["negation"][index])]\n\n    for label, name in enumerate(TARGET_SYMPTOMS):\n        fn = np.flatnonzero((targets[:, label] == 1) & (predictions[:, label] == 0)).tolist()\n        fp = np.flatnonzero((targets[:, label] == 0) & (predictions[:, label] == 1)).tolist()\n        indices = []\n        while (fn or fp) and len(indices) < max_per_label:\n            for pending in (fn, fp):\n                if pending and len(indices) < max_per_label:\n                    indices.append(pending.pop(0))\n        output = output_dir / f"errors_{name}.csv"\n\n        def write(handle):\n            writer = csv.writer(handle)\n            writer.writerow(header)\n            for index in indices:\n                writer.writerow(review_row(index, name, "FN" if targets[index, label] else "FP"))\n        _atomic_text(output, write)\n        outputs.append(output)\n\n    nausea_index, vomiting_index = TARGET_SYMPTOMS.index("오심"), TARGET_SYMPTOMS.index("구토")\n    nausea, vomiting = targets[:, nausea_index].astype(bool), targets[:, vomiting_index].astype(bool)\n    combinations = {"neither": ~nausea & ~vomiting, "nausea_only": nausea & ~vomiting,\n                    "vomiting_only": ~nausea & vomiting, "both": nausea & vomiting}\n    group_counts = {}\n    for name, mask in combinations.items():\n        row = {"n": int(mask.sum())}\n        for symptom, column in (("nausea", nausea_index), ("vomiting", vomiting_index)):\n            actual, predicted = targets[:, column].astype(bool), predictions[:, column].astype(bool)\n            row[f"predicted_{symptom}"] = int((mask & predicted).sum())\n            row[f"{symptom}_tp"] = int((mask & actual & predicted).sum())\n            row[f"{symptom}_fp"] = int((mask & ~actual & predicted).sum())\n            row[f"{symptom}_fn"] = int((mask & actual & ~predicted).sum())\n            row[f"{symptom}_tn"] = int((mask & ~actual & ~predicted).sum())\n        group_counts[name] = row\n    summary_path = output_dir / "nausea_vomiting_error_groups.json"\n    write_json(summary_path, group_counts)\n    outputs.append(summary_path)\n\n    predicted_nausea = predictions[:, nausea_index].astype(bool)\n    controls = {"TP": nausea & predicted_nausea, "FP": ~nausea & predicted_nausea,\n                "FN": nausea & ~predicted_nausea, "TN": ~nausea & ~predicted_nausea}\n    rng = np.random.default_rng(42)\n    sampled = []\n    for category, mask in controls.items():\n        candidates = np.flatnonzero(mask)\n        chosen = rng.choice(candidates, size=min(10, len(candidates)), replace=False)\n        sampled.extend((int(index), category) for index in chosen)\n    # Mix categories for human review while preserving reproducibility.\n    rng.shuffle(sampled)\n    review_path = output_dir / "review_오심.csv"\n\n    def write_controls(handle):\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        for index, category in sampled:\n            writer.writerow(review_row(index, "오심", category))\n    _atomic_text(review_path, write_controls)\n    outputs.append(review_path)\n    return outputs\n', 'bootstrap.py': '"""Safely restore label JSONs only; old model weights are never extracted."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport tempfile\nimport zipfile\n\nTRAIN_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Training/2.라벨링데이터/TL_서울_구급/"\nVALID_PREFIX = "DCC_AIdeas/DATA/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급/"\nMARKER_NAME = ".mission3_label_bundle_complete.json"\n\n\ndef _relative(name):\n    path = PurePosixPath(name)\n    if (not isinstance(name, str) or not name or "\\\\" in name or ":" in name\n            or path.is_absolute() or ".." in path.parts\n            or path.as_posix() != name.rstrip("/")):\n        raise ValueError(f"Unsafe bundle path: {name}")\n    return Path(*path.parts)\n\n\ndef _target(root, name):\n    target = root / _relative(name)\n    if not target.resolve().is_relative_to(root):\n        raise ValueError(f"Bundle path escapes extraction directory: {name}")\n    # Existing links must not redirect even to another location inside the root.\n    for component in (target, *target.parents):\n        if component == root.parent:\n            break\n        if component.is_symlink():\n            raise ValueError(f"Extraction destination contains a symlink: {name}")\n    return target\n\n\ndef _hash(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as reader:\n        for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _selected_files(archive):\n    infos = archive.infolist()\n    names = set()\n    entry_types = {}\n    for info in infos:\n        # Windows ZipInfo normalizes backslashes in filename; orig_filename\n        # retains the stored member spelling that must also pass validation.\n        _relative(info.orig_filename)\n        _relative(info.filename)\n        normalized = info.filename.rstrip("/").casefold()\n        if normalized in names:\n            raise ValueError(f"Duplicate ZIP member or directory: {info.filename}")\n        names.add(normalized)\n        entry_types[normalized] = info.is_dir()\n        if stat.S_ISLNK(info.external_attr >> 16) or info.flag_bits & 1:\n            raise ValueError("Links and encrypted ZIP entries are not supported")\n    for name in entry_types:\n        if any(parent.as_posix() in entry_types and not entry_types[parent.as_posix()]\n               for parent in PurePosixPath(name).parents if parent.as_posix() != "."):\n            raise ValueError("ZIP file/directory collision")\n    selected = [info for info in infos if not info.is_dir() and info.filename.endswith(".json")\n                and info.filename.startswith((TRAIN_PREFIX, VALID_PREFIX))]\n    for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n        subset = [info for info in selected if info.filename.startswith(prefix)]\n        if not subset:\n            raise ValueError(f"Input ZIP has no label JSONs in {prefix}")\n        if len({PurePosixPath(info.filename).name.casefold() for info in subset}) != len(subset):\n            raise ValueError("Duplicate label JSON filenames in input ZIP")\n    by_name = {info.filename: info for info in selected}\n    if "bundle_manifest.json" in archive.namelist():\n        raw = archive.read("bundle_manifest.json")\n        manifest = json.loads(raw)\n        if not isinstance(manifest, dict) or manifest.get("format") != "mission3_ver5_inputs_v1":\n            raise ValueError("Unsupported Mission 3 input bundle manifest")\n        files = manifest.get("files")\n        if not isinstance(files, list):\n            raise ValueError("Bundle manifest has no files")\n        metadata, listed = {}, set()\n        for row in files:\n            if not isinstance(row, dict) or not isinstance(row.get("path"), str):\n                raise ValueError("Invalid bundle manifest file entry")\n            name = row["path"]\n            _relative(name)\n            if name.casefold() in listed:\n                raise ValueError(f"Duplicate manifest entry: {name}")\n            listed.add(name.casefold())\n            if name not in by_name:\n                continue\n            size, digest = row.get("size"), row.get("sha256")\n            if (type(size) is not int or size < 0 or not isinstance(digest, str)\n                    or re.fullmatch(r"[0-9a-f]{64}", digest) is None):\n                raise ValueError(f"Invalid manifest metadata: {name}")\n            if by_name[name].file_size != size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest}\n        if set(metadata) != set(by_name):\n            raise ValueError("Bundle manifest does not list every label JSON")\n        for prefix, key in ((TRAIN_PREFIX, "training_count"), (VALID_PREFIX, "validation_count")):\n            count = sum(name.startswith(prefix) for name in metadata)\n            if type(manifest.get(key)) is not int or manifest[key] != count:\n                raise ValueError(f"Bundle JSON count mismatch: {key}")\n        signature = hashlib.sha256(raw).hexdigest()\n    else:\n        # Without a manifest, hash only label contents, never the large old models.\n        metadata = {}\n        for name, info in sorted(by_name.items()):\n            digest, size = hashlib.sha256(), 0\n            with archive.open(info) as reader:\n                for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                    digest.update(chunk)\n                    size += len(chunk)\n            if size != info.file_size:\n                raise ValueError(f"Bundle size mismatch: {name}")\n            metadata[name] = {"path": name, "size": size, "sha256": digest.hexdigest()}\n        signature = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()\n    return [metadata[name] for name in sorted(metadata)], signature\n\n\ndef _validate_directory(directory, name):\n    directory = Path(directory).expanduser().resolve()\n    if not directory.is_dir() or not any(path.is_file() for path in directory.rglob("*.json")):\n        raise ValueError(f"{name} label directory does not contain JSON files: {directory}")\n    return directory\n\n\ndef prepare_data(input_zip, work_dir, train_dir="", validation_dir=""):\n    """Use two explicit label directories, or verify/extract the existing bundle.\n\n    Reuse requires a completion marker plus fresh content hashes. This detects\n    changed files even when their size and modification time are preserved.\n    """\n    if bool(train_dir) != bool(validation_dir):\n        raise ValueError("Set both train_dir and validation_dir, or leave both empty")\n    if train_dir:\n        train = _validate_directory(train_dir, "Training")\n        valid = _validate_directory(validation_dir, "Validation")\n        if train == valid or train.is_relative_to(valid) or valid.is_relative_to(train):\n            raise ValueError("Training and Validation label directories must be distinct and disjoint")\n        return train, valid\n    archive_path = Path(input_zip).expanduser().resolve()\n    if not archive_path.is_file():\n        raise FileNotFoundError(f"Input ZIP is missing: {archive_path}")\n    requested_root = Path(work_dir).expanduser()\n    if requested_root.is_symlink():\n        raise ValueError("Extraction work_dir must not be a symlink")\n    root = requested_root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    marker = root / MARKER_NAME\n    with zipfile.ZipFile(archive_path) as archive:\n        files, signature = _selected_files(archive)\n        # Resolve every destination before any extraction, including existing links.\n        destinations = {row["path"]: _target(root, row["path"]) for row in files}\n        expected_paths = set(destinations.values())\n        for prefix in (TRAIN_PREFIX, VALID_PREFIX):\n            directory = root / prefix\n            if directory.exists():\n                unexpected = set(directory.rglob("*.json")) - expected_paths\n                if unexpected:\n                    raise ValueError("Extraction directory contains label JSONs from another input bundle")\n        previous = None\n        if marker.exists():\n            if marker.is_symlink():\n                raise ValueError("Completion marker must not be a symlink")\n            try:\n                previous = json.loads(marker.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                previous = None\n        expected = {row["path"]: {"size": row["size"], "sha256": row["sha256"]} for row in files}\n        reuse = (isinstance(previous, dict) and previous.get("signature") == signature\n                 and previous.get("files") == expected)\n        pending = []\n        for row in files:\n            target = destinations[row["path"]]\n            verified = (target.is_file() and target.stat().st_size == row["size"]\n                        and _hash(target) == row["sha256"])\n            if not verified:\n                pending.append(row)\n        if reuse and not pending:\n            return root / TRAIN_PREFIX, root / VALID_PREFIX\n        if marker.exists():\n            marker.unlink()\n        if shutil.disk_usage(root).free < sum(row["size"] for row in pending):\n            raise OSError("Insufficient local disk space to extract label JSONs")\n        for row in pending:\n            target = destinations[row["path"]]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            temporary = None\n            try:\n                digest, size = hashlib.sha256(), 0\n                with archive.open(row["path"]) as reader, tempfile.NamedTemporaryFile(\n                        dir=target.parent, prefix=".extract_", suffix=".tmp", delete=False) as writer:\n                    temporary = Path(writer.name)\n                    for chunk in iter(lambda: reader.read(1024 * 1024), b""):\n                        writer.write(chunk)\n                        digest.update(chunk)\n                        size += len(chunk)\n                if size != row["size"] or digest.hexdigest() != row["sha256"]:\n                    raise ValueError(f"ZIP content verification failed: {row[\'path\']}")\n                temporary.replace(target)\n            finally:\n                if temporary is not None:\n                    temporary.unlink(missing_ok=True)\n        # Do not publish an incomplete extraction as a reusable data set.\n        temporary = None\n        try:\n            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=root,\n                                             prefix=".marker_", suffix=".tmp", delete=False) as writer:\n                temporary = Path(writer.name)\n                json.dump({"signature": signature, "files": expected}, writer, ensure_ascii=False)\n            temporary.replace(marker)\n        finally:\n            if temporary is not None:\n                temporary.unlink(missing_ok=True)\n    return root / TRAIN_PREFIX, root / VALID_PREFIX\n', 'budget.py': '"""Share a wall-clock deadline across setup, training, and final export.\n\nCreate the monotonic deadline once per notebook invocation, before package\ninstallation. Do not restore a monotonic timestamp from an earlier Colab VM.\n``Budget`` reserves finalization time when ``training=True``; subprocess\nsupervision always enforces the actual invocation deadline.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom numbers import Real\nimport os\nimport signal\nimport subprocess\nimport time\n\n\nclass BudgetExceeded(TimeoutError):\n    """Work stopped without declaring an incomplete artifact complete."""\n\n    def __init__(self, stage, *, deadline_monotonic=None):\n        self.stage = str(stage)\n        self.deadline_monotonic = deadline_monotonic\n        super().__init__(f"Wall-clock budget exhausted during {self.stage}")\n\n\ndef _finite_nonnegative(value, name):\n    if isinstance(value, bool) or not isinstance(value, Real):\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    value = float(value)\n    if not math.isfinite(value) or value < 0:\n        raise ValueError(f"{name} must be a finite nonnegative number")\n    return value\n\n\nclass Budget:\n    """A deadline with conservative admission checks for expensive stages.\n\n    ``remaining(training=True)`` subtracts the export/evaluation reserve.\n    ``check`` raises at an exhausted deadline and otherwise returns seconds left.\n    ``can_start`` applies the safety factor to an estimated duration; it does not\n    extend the deadline or admit work when no time remains.\n    """\n\n    def __init__(self, deadline_monotonic, finalization_reserve_s=1200,\n                 safety_factor=1.25, clock=time.monotonic):\n        self.deadline_monotonic = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n        self.finalization_reserve_s = _finite_nonnegative(finalization_reserve_s, "finalization_reserve_s")\n        self.safety_factor = _finite_nonnegative(safety_factor, "safety_factor")\n        if self.safety_factor < 1:\n            raise ValueError("safety_factor must be at least one")\n        if not callable(clock):\n            raise ValueError("clock must be callable")\n        self.clock = clock\n\n    def remaining(self, training=False):\n        if not isinstance(training, bool):\n            raise ValueError("training must be a boolean")\n        now = _finite_nonnegative(self.clock(), "clock result")\n        reserve = self.finalization_reserve_s if training else 0\n        return max(0.0, self.deadline_monotonic - reserve - now)\n\n    def check(self, stage, training=False):\n        remaining = self.remaining(training=training)\n        if remaining <= 0:\n            raise BudgetExceeded(stage, deadline_monotonic=self.deadline_monotonic)\n        return remaining\n\n    def can_start(self, estimated_seconds, training=True):\n        estimate = _finite_nonnegative(estimated_seconds, "estimated_seconds")\n        remaining = self.remaining(training=training)\n        return remaining > 0 and estimate * self.safety_factor <= remaining\n\n\ndef _signal_process(process, force=False):\n    """Signal only this supervisor\'s new process group (or Windows child)."""\n    try:\n        if os.name == "posix":\n            # Popen(start_new_session=True) makes this child\'s PID its group ID.\n            # Signal the group even if its leader exited, to stop descendants.\n            os.killpg(process.pid, signal.SIGKILL if force else signal.SIGTERM)\n        elif process.poll() is None:\n            process.kill() if force else process.terminate()\n    except ProcessLookupError:\n        pass\n\n\ndef _stop_process(process, deadline_monotonic):\n    """Use at most the remaining shutdown margin for graceful stop and reaping."""\n    _signal_process(process)\n    remaining = max(0.0, deadline_monotonic - time.monotonic())\n    try:\n        process.wait(timeout=min(5.0, remaining / 2))\n    except subprocess.TimeoutExpired:\n        pass\n    # A descendant may ignore SIGTERM even when the group leader has exited.\n    _signal_process(process, force=True)\n    try:\n        process.wait(timeout=max(0.0, deadline_monotonic - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        # SIGKILL/TerminateProcess has already been issued; never wait unbounded.\n        pass\n\n\ndef run_supervised(command, deadline_monotonic, shutdown_margin_s=10, cwd=None, env=None):\n    """Run a command with inherited output and a bounded wall-clock watchdog.\n\n    Returns zero on success. Nonzero exit raises ``CalledProcessError``; timeout\n    raises ``BudgetExceeded`` after terminating the child. POSIX descendants are\n    stopped as a group; Windows supervision stops only its own direct child.\n    The shutdown margin is inside the supplied deadline, not added after it.\n    Commands must be argument sequences and are never executed through a shell.\n    """\n    deadline = _finite_nonnegative(deadline_monotonic, "deadline_monotonic")\n    margin = _finite_nonnegative(shutdown_margin_s, "shutdown_margin_s")\n    if margin <= 0:\n        raise ValueError("shutdown_margin_s must be positive")\n    if isinstance(command, (str, bytes)) or not isinstance(command, (list, tuple)) or not command:\n        raise ValueError("command must be a nonempty argument sequence")\n    if deadline - margin - time.monotonic() <= 0:\n        raise BudgetExceeded("subprocess startup", deadline_monotonic=deadline)\n    process = subprocess.Popen(command, cwd=cwd, env=env, start_new_session=os.name == "posix")\n    try:\n        returncode = process.wait(timeout=max(0.0, deadline - margin - time.monotonic()))\n    except subprocess.TimeoutExpired:\n        _stop_process(process, deadline)\n        raise BudgetExceeded("subprocess execution", deadline_monotonic=deadline) from None\n    except BaseException:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise\n    if returncode:\n        _stop_process(process, min(deadline, time.monotonic() + margin))\n        raise subprocess.CalledProcessError(returncode, command)\n    return returncode\n', 'trainer.py': '"""Matched BERT/RoBERTa fine-tuning with fixed multilabel decisions.\n\nOnly caller-provided Training rows enter the loss. The runtime owns the split,\ncandidate selection, epoch count, admission checks, and full-data refit.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import nullcontext\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport re\nimport tempfile\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer, get_linear_schedule_with_warmup\n\nfrom .data import TARGET_SYMPTOMS\n\n\nFORMAT = "mission3_ver24_mean_bce_v1"\nDECISION = "independent_logits_greater_than_or_equal_to_zero"\n\n\ndef seed_everything(seed=42):\n    random.seed(int(seed))\n    np.random.seed(int(seed))\n    torch.manual_seed(int(seed))\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(int(seed))\n\n\ndef _targets(values, rows=None):\n    values = np.asarray(values)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or (rows is not None and len(values) != rows)\n            or not np.isin(values, (0, 1)).all()):\n        raise ValueError("Targets must be a binary (samples, 9) array")\n    return values.astype(np.float32, copy=False)\n\n\ndef positive_weights(fit_targets):\n    """Fixed sqrt5 rule, calculated exclusively from the supplied fit rows."""\n    values = _targets(fit_targets)\n    if not len(values):\n        raise ValueError("Positive weights require at least one fit row")\n    positives = values.sum(axis=0, dtype=np.float64)\n    ratios = np.maximum((len(values) - positives) / np.maximum(positives, 1), 1)\n    return torch.tensor(np.clip(np.sqrt(ratios), 1, 5), dtype=torch.float32)\n\n\ndef fixed_decisions(logits):\n    values = np.asarray(logits)\n    if (values.ndim != 2 or values.shape[1] != len(TARGET_SYMPTOMS)\n            or not np.isfinite(values).all()):\n        raise ValueError("Logits must be finite with shape (samples, 9)")\n    return (values >= 0).astype(np.int8)\n\n\nclass MeanPoolClassifier(nn.Module):\n    def __init__(self, encoder, dropout=.1):\n        super().__init__()\n        if encoder.config.model_type not in ("bert", "roberta"):\n            raise ValueError("Only standard BERT and RoBERTa encoders are supported")\n        if not 0 <= float(dropout) < 1:\n            raise ValueError("Dropout must be in [0, 1)")\n        self.encoder = encoder\n        # Mean pooling uses last_hidden_state. A pretrained MLM has no pooler\n        # weights, and an unused randomly initialized pooler adds no signal.\n        if hasattr(self.encoder, "pooler"):\n            self.encoder.pooler = None\n        self.dropout_probability = float(dropout)\n        self.dropout = nn.Dropout(self.dropout_probability)\n        self.classifier = nn.Linear(encoder.config.hidden_size, len(TARGET_SYMPTOMS))\n\n    def forward(self, input_ids, attention_mask, token_type_ids=None):\n        inputs = {"input_ids": input_ids, "attention_mask": attention_mask}\n        if token_type_ids is not None:\n            inputs["token_type_ids"] = token_type_ids\n        hidden = self.encoder(**inputs).last_hidden_state\n        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)\n        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)\n        return self.classifier(self.dropout(pooled))\n\n\ndef create_model(model_name, revision=None, *, seed=42, cache_dir=None,\n                 local_files_only=False, dropout=.1):\n    """Start from a pinned public encoder, or an explicitly local test snapshot."""\n    local = Path(model_name).is_dir()\n    if not local and (not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision)):\n        raise ValueError("Remote pretrained models require an immutable 40-character revision")\n    seed_everything(seed)\n    options = dict(trust_remote_code=False, local_files_only=local_files_only)\n    if not local:\n        options["revision"] = revision\n    if cache_dir is not None:\n        options["cache_dir"] = str(cache_dir)\n    tokenizer = AutoTokenizer.from_pretrained(str(model_name), use_fast=True, **options)\n    encoder = AutoModel.from_pretrained(str(model_name), add_pooling_layer=False, **options)\n    if tokenizer.pad_token_id is None:\n        raise ValueError("The tokenizer must provide a padding token")\n    if len(tokenizer) != encoder.config.vocab_size:\n        raise ValueError("Tokenizer and pretrained encoder vocabulary sizes differ")\n    model = MeanPoolClassifier(encoder, dropout=dropout)\n    return model, tokenizer\n\n\nclass _Rows(Dataset):\n    def __init__(self, texts, targets=None, indices=None):\n        if not len(texts) or any(not isinstance(text, str) for text in texts):\n            raise ValueError("Texts must be a nonempty sequence of strings")\n        self.texts = texts\n        self.targets = None if targets is None else _targets(targets, len(texts))\n        self.indices = (np.arange(len(texts), dtype=np.int64) if indices is None\n                        else np.asarray(indices))\n        if (self.indices.ndim != 1 or not len(self.indices)\n                or not np.issubdtype(self.indices.dtype, np.integer)\n                or np.any(self.indices < 0) or np.any(self.indices >= len(texts))\n                or len(np.unique(self.indices)) != len(self.indices)):\n            raise ValueError("Row indices must be unique valid integers")\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, index):\n        row = int(self.indices[index])\n        return self.texts[row], None if self.targets is None else self.targets[row]\n\n\nclass _Collate:\n    def __init__(self, tokenizer, max_length):\n        self.tokenizer = tokenizer\n        self.max_length = max_length\n\n    def __call__(self, rows):\n        tokens = self.tokenizer([row[0] for row in rows], padding=True,\n                                truncation=True, max_length=self.max_length,\n                                return_tensors="pt")\n        result = {name: value for name, value in tokens.items()\n                  if name in ("input_ids", "attention_mask", "token_type_ids")}\n        if rows[0][1] is not None:\n            result["labels"] = torch.tensor(np.stack([row[1] for row in rows]), dtype=torch.float32)\n        return result\n\n\ndef make_loader(tokenizer, texts, targets=None, indices=None, *, batch_size=8,\n                max_length=512, shuffle=False, seed=42, pin_memory=False):\n    if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:\n        raise ValueError("Batch size must be a positive integer")\n    if isinstance(max_length, bool) or not isinstance(max_length, int) or not 2 <= max_length <= 512:\n        raise ValueError("Max length must be an integer in [2, 512]")\n    generator = torch.Generator().manual_seed(int(seed))\n    return DataLoader(_Rows(texts, targets, indices), batch_size=batch_size,\n                      shuffle=bool(shuffle), generator=generator,\n                      collate_fn=_Collate(tokenizer, max_length), drop_last=False,\n                      num_workers=0, pin_memory=bool(pin_memory))\n\n\ndef build_optimizer(model, lr=2e-5, weight_decay=.01):\n    if not math.isfinite(lr) or lr <= 0 or not math.isfinite(weight_decay) or weight_decay < 0:\n        raise ValueError("Invalid optimizer settings")\n    decay, no_decay = [], []\n    for name, parameter in model.named_parameters():\n        if parameter.requires_grad:\n            (no_decay if name.endswith("bias") or "LayerNorm.weight" in name else decay).append(parameter)\n    return torch.optim.AdamW([\n        {"params": decay, "weight_decay": float(weight_decay)},\n        {"params": no_decay, "weight_decay": 0.0},\n    ], lr=float(lr))\n\n\ndef build_scheduler(optimizer, total_updates, warmup_ratio=.1):\n    if (isinstance(total_updates, bool) or not isinstance(total_updates, int)\n            or total_updates < 1 or not 0 <= warmup_ratio < 1):\n        raise ValueError("Invalid learning-rate schedule")\n    return get_linear_schedule_with_warmup(\n        optimizer, int(total_updates * warmup_ratio), total_updates)\n\n\ndef _amp_dtype(device):\n    if torch.device(device).type != "cuda":\n        return None\n    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n\n\ndef make_scaler(device):\n    device = torch.device(device)\n    return torch.amp.GradScaler("cuda", enabled=_amp_dtype(device) == torch.float16)\n\n\ndef _autocast(device):\n    dtype = _amp_dtype(device)\n    return torch.autocast("cuda", dtype=dtype) if dtype is not None else nullcontext()\n\n\ndef _synchronize(device):\n    if torch.device(device).type == "cuda":\n        torch.cuda.synchronize(device)\n\n\ndef train_epoch(model, loader, optimizer, *, device, pos_weight, accumulation_steps=2,\n                scheduler=None, scaler=None, budget=None, on_update=None, max_updates=None):\n    """Train all rows once, including the final partial accumulation group.\n\n    A pilot can stop after ``max_updates`` successful updates. Such a partial\n    epoch is explicitly marked incomplete and must never be a final refit.\n    Timing covers tokenization, transfer, forward, backward, and optimizer work.\n    """\n    if (isinstance(accumulation_steps, bool) or not isinstance(accumulation_steps, int)\n            or accumulation_steps < 1):\n        raise ValueError("Accumulation steps must be a positive integer")\n    if max_updates is not None and (isinstance(max_updates, bool)\n                                    or not isinstance(max_updates, int) or max_updates < 1):\n        raise ValueError("Max updates must be a positive integer")\n    if not len(loader):\n        raise ValueError("Training loader is empty")\n    device = torch.device(device)\n    weights = torch.as_tensor(pos_weight, dtype=torch.float32, device=device)\n    if weights.shape != (len(TARGET_SYMPTOMS),) or not torch.isfinite(weights).all() or (weights <= 0).any():\n        raise ValueError("Positive weights must contain nine finite positive numbers")\n    scaler = make_scaler(device) if scaler is None else scaler\n    loss_function = nn.BCEWithLogitsLoss(pos_weight=weights)\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    successful = attempted = examples = seen_batches = group_rows = 0\n    loss_sum = 0.0\n    update_seconds = []\n    _synchronize(device)\n    started = previous_update = time.monotonic()\n    try:\n        for batch_index, batch in enumerate(loader):\n            if budget is not None:\n                budget.check("encoder_training", training=True)\n            if "labels" not in batch:\n                raise ValueError("Training batches must contain labels")\n            labels = batch["labels"].to(device, non_blocking=True)\n            inputs = {key: value.to(device, non_blocking=True) for key, value in batch.items()\n                      if key != "labels"}\n            with _autocast(device):\n                logits = model(**inputs)\n                loss = loss_function(logits.float(), labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Training loss is not finite")\n            rows = len(labels)\n            # Weight microbatches by row count before dividing accumulated\n            # gradients; the final short batch must not be overrepresented.\n            scaler.scale(loss * rows).backward()\n            group_rows += rows\n            examples += rows\n            seen_batches += 1\n            loss_sum += float(loss.detach()) * rows\n            if (batch_index + 1) % accumulation_steps and batch_index + 1 < len(loader):\n                continue\n            scaler.unscale_(optimizer)\n            for parameter in model.parameters():\n                if parameter.grad is not None:\n                    parameter.grad.div_(group_rows)\n            norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            if not scaler.is_enabled() and not torch.isfinite(norm):\n                raise FloatingPointError("Training gradients are not finite")\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            attempted += 1\n            accepted = scaler.get_scale() >= old_scale\n            optimizer.zero_grad(set_to_none=True)\n            group_rows = 0\n            _synchronize(device)\n            now = time.monotonic()\n            if accepted:\n                successful += 1\n                update_seconds.append(now - previous_update)\n                if scheduler is not None:\n                    scheduler.step()\n                progress = {"successful_updates": successful, "attempted_updates": attempted,\n                            "examples": examples, "batches": seen_batches,\n                            "elapsed_seconds": now - started,\n                            "update_seconds": list(update_seconds),\n                            "measured_update_seconds": list(update_seconds[1:])}\n                if on_update is not None:\n                    on_update(progress)\n                if max_updates is not None and successful >= max_updates:\n                    break\n                previous_update = now\n            # An overflow\'s duration remains in the next successful update\'s\n            # timing, so admission estimates do not hide skipped work.\n    finally:\n        optimizer.zero_grad(set_to_none=True)\n    _synchronize(device)\n    if successful == 0:\n        raise FloatingPointError("Training completed no successful optimizer updates")\n    measured = update_seconds[1:]\n    return {"successful_updates": successful, "attempted_updates": attempted,\n            "examples": examples, "batches": seen_batches,\n            "mean_loss": loss_sum / max(examples, 1),\n            "elapsed_seconds": time.monotonic() - started,\n            "update_seconds": update_seconds, "measured_update_seconds": measured,\n            "seconds_per_update": float(np.mean(measured)) if measured else None,\n            "completed_epoch": seen_batches == len(loader)}\n\n\n@torch.no_grad()\ndef predict_logits(model, loader, *, device, budget=None, max_batches=None, return_stats=False):\n    if max_batches is not None and (isinstance(max_batches, bool)\n                                    or not isinstance(max_batches, int) or max_batches < 1):\n        raise ValueError("Max batches must be a positive integer")\n    device = torch.device(device)\n    model.eval()\n    chunks = []\n    _synchronize(device)\n    started = time.monotonic()\n    for index, batch in enumerate(loader):\n        if budget is not None:\n            budget.check("encoder_inference", training=False)\n        inputs = {key: value.to(device, non_blocking=True) for key, value in batch.items()\n                  if key != "labels"}\n        with _autocast(device):\n            logits = model(**inputs)\n        values = logits.float().cpu().numpy()\n        fixed_decisions(values)\n        chunks.append(values)\n        if max_batches is not None and index + 1 >= max_batches:\n            break\n    if not chunks:\n        raise ValueError("Prediction loader is empty")\n    _synchronize(device)\n    values = np.concatenate(chunks, axis=0)\n    elapsed = time.monotonic() - started\n    if return_stats:\n        return {"logits": values, "examples": len(values), "elapsed_seconds": elapsed,\n                "seconds_per_example": elapsed / len(values)}\n    return values\n\n\ndef _json_metadata(metadata):\n    copied = json.loads(json.dumps(metadata, ensure_ascii=False, allow_nan=False))\n    if not isinstance(copied, dict):\n        raise ValueError("Checkpoint metadata must be an object")\n    return copied\n\n\ndef save_checkpoint(path, model, metadata):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = {"format": FORMAT, "target_symptoms": list(TARGET_SYMPTOMS),\n               "decision": DECISION, "encoder_config": model.encoder.config.to_dict(),\n               "dropout": model.dropout_probability, "metadata": _json_metadata(metadata),\n               "state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()}}\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(dir=path.parent, prefix=".checkpoint_", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n        torch.save(payload, temporary)\n        temporary.replace(path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef load_checkpoint(path, device="cpu"):\n    payload = torch.load(Path(path), map_location="cpu", weights_only=True)\n    if (payload.get("format") != FORMAT or payload.get("target_symptoms") != list(TARGET_SYMPTOMS)\n            or payload.get("decision") != DECISION):\n        raise ValueError("Checkpoint format, label order, or fixed decision policy is invalid")\n    config_dict = dict(payload["encoder_config"])\n    kind = config_dict.pop("model_type")\n    if kind not in ("bert", "roberta"):\n        raise ValueError("Checkpoint encoder type is unsupported")\n    config = AutoConfig.for_model(kind, **config_dict)\n    model = MeanPoolClassifier(AutoModel.from_config(\n        config, trust_remote_code=False, add_pooling_layer=False), payload["dropout"])\n    model.load_state_dict(payload["state_dict"], strict=True)\n    return model.to(device).eval(), _json_metadata(payload["metadata"])\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef export_model(directory, model, tokenizer, metadata):\n    """Write a standalone checkpoint and tokenizer; no remote code is required."""\n    directory = Path(directory)\n    if directory.exists() and any(directory.iterdir()):\n        raise FileExistsError("Export destination must be empty")\n    directory.mkdir(parents=True, exist_ok=True)\n    meta = _json_metadata(metadata)\n    max_length = meta.get("max_length", 512)\n    if isinstance(max_length, bool) or not isinstance(max_length, int) or not 2 <= max_length <= 512:\n        raise ValueError("Export max_length must be an integer in [2, 512]")\n    meta["max_length"] = max_length\n    tokenizer.save_pretrained(directory / "tokenizer")\n    save_checkpoint(directory / "model.pt", model, meta)\n    hashes = {path.relative_to(directory).as_posix(): _sha256(path)\n              for path in sorted(directory.rglob("*")) if path.is_file()}\n    manifest = {"format": FORMAT, "target_symptoms": list(TARGET_SYMPTOMS),\n                "decision": DECISION, "files": hashes, "metadata": meta}\n    (directory / "manifest.json").write_text(\n        json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    return manifest\n\n\ndef load_export(directory, device="cpu"):\n    """Verify every exported asset before local-only tokenizer/model loading."""\n    directory = Path(directory).resolve()\n    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))\n    if (manifest.get("format") != FORMAT or manifest.get("target_symptoms") != list(TARGET_SYMPTOMS)\n            or manifest.get("decision") != DECISION or not isinstance(manifest.get("files"), dict)):\n        raise ValueError("Export manifest is invalid")\n    if "model.pt" not in manifest["files"]:\n        raise ValueError("Export checkpoint is missing")\n    for relative, expected in manifest["files"].items():\n        path = directory / relative\n        if (not isinstance(relative, str) or path.is_symlink()\n                or not path.resolve().is_relative_to(directory) or not path.is_file()\n                or _sha256(path) != expected):\n            raise ValueError("Export asset integrity check failed")\n    actual = {path.relative_to(directory).as_posix() for path in directory.rglob("*")\n              if path.is_file() and path.name != "manifest.json"}\n    if actual != set(manifest["files"]):\n        raise ValueError("Export includes unverified files")\n    model, metadata = load_checkpoint(directory / "model.pt", device=device)\n    if metadata != manifest.get("metadata"):\n        raise ValueError("Export checkpoint and manifest metadata disagree")\n    tokenizer = AutoTokenizer.from_pretrained(\n        directory / "tokenizer", local_files_only=True, trust_remote_code=False, use_fast=True)\n    if tokenizer.pad_token_id is None or len(tokenizer) != model.encoder.config.vocab_size:\n        raise ValueError("Export tokenizer and model are incompatible")\n    return model, tokenizer, metadata\n\n\ndef predict_texts(model, tokenizer, texts, *, device="cpu", batch_size=16,\n                  max_length=512, budget=None):\n    loader = make_loader(tokenizer, texts, batch_size=batch_size, max_length=max_length,\n                         pin_memory=torch.device(device).type == "cuda")\n    return predict_logits(model, loader, device=device, budget=budget)\n', 'runtime.py': '"""Budgeted, matched encoder comparison using untouched Training labels."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport uuid\n\nimport numpy as np\n\nfrom . import data, trainer\nfrom .bootstrap import prepare_data\nfrom .budget import Budget, BudgetExceeded\n\nORDER = ("medical", "control")\nNAUSEA = data.TARGET_SYMPTOMS.index("오심")\n\n\nclass _TrainingBudget:\n    """Keep internal inference out of the final export/evaluation reserve."""\n    def __init__(self, budget):\n        self.budget = budget\n\n    def check(self, stage, training=False):\n        return self.budget.check(stage, training=True)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8-sig"))\n\n\ndef validate_settings(settings):\n    result = dict(settings)\n    pinned = {"version": "ver24", "seed": 42, "holdout_size": .2, "max_length": 512,\n              "learning_rate": 2e-5, "weight_decay": .01, "dropout": .1, "warmup_ratio": .1}\n    for key, value in pinned.items():\n        if type(result.get(key)) is not type(value) or result[key] != value:\n            raise ValueError("Pinned experiment setting changed: " + key)\n    for key in ("batch_size", "gradient_accumulation", "max_epochs", "pilot_optimizer_steps", "log_every"):\n        if type(result.get(key)) is not int or result[key] < 1:\n            raise ValueError(key + " must be a positive integer")\n    if result["max_epochs"] > 3 or result["pilot_optimizer_steps"] < 10:\n        raise ValueError("Use at most three epochs and at least ten measured pilot updates")\n    for key in ("time_budget_minutes", "finalization_reserve_seconds", "budget_safety_factor"):\n        value = result.get(key)\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:\n            raise ValueError(key + " must be finite and positive")\n    if (result["time_budget_minutes"] > 180 or result["budget_safety_factor"] < 1.25\n            or result["finalization_reserve_seconds"] < 600\n            or result["finalization_reserve_seconds"] >= result["time_budget_minutes"] * 60):\n        raise ValueError("Use at most 180 minutes, sufficient export reserve, and a conservative safety factor")\n    for key in ("allow_cpu", "run_validation"):\n        if type(result.get(key)) is not bool:\n            raise ValueError(key + " must be boolean")\n    for key in ("input_zip", "output_dir", "work_dir", "train_dir", "validation_dir"):\n        if not isinstance(result.get(key), str):\n            raise ValueError(key + " must be a path string")\n    if any(any(word in key.lower() for word in ("threshold", "calibration", "logit_shift", "label_mask"))\n           for key in result):\n        raise ValueError("Post-hoc decision changes and label masks are not supported")\n    return result\n\n\ndef dataset_digest(texts, targets):\n    digest = hashlib.sha256()\n    for text, labels in zip(texts, targets):\n        digest.update(text.encode("utf-8"))\n        digest.update(b"\\0")\n        digest.update(np.asarray(labels, dtype=np.int8).tobytes())\n    return digest.hexdigest()\n\n\ndef _hash(path, budget=None):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):\n            if budget:\n                budget.check("artifact hash")\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _save_npz(path, **arrays):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, **arrays)\n    temporary.replace(path)\n\n\ndef _release():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _device(settings):\n    import torch\n    if torch.cuda.is_available():\n        return "cuda"\n    if not settings["allow_cpu"]:\n        raise RuntimeError("Colab에서 GPU 런타임을 선택하세요")\n    return "cpu"\n\n\ndef _create(spec, settings, device):\n    model, tokenizer = trainer.create_model(spec["model_id"], spec["revision"], seed=settings["seed"],\n        cache_dir=str(Path(settings["work_dir"]) / "hf_cache"), dropout=settings["dropout"])\n    model.to(device)\n    return model, tokenizer\n\n\ndef _loader(tokenizer, texts, targets, indices, settings, *, shuffle=False, device="cpu"):\n    return trainer.make_loader(tokenizer, texts, targets=targets, indices=indices,\n        batch_size=settings["batch_size"], max_length=settings["max_length"], shuffle=shuffle,\n        seed=settings["seed"], pin_memory=device == "cuda")\n\n\ndef _optimizer(model, loader, epochs, settings, device):\n    optimizer = trainer.build_optimizer(model, lr=settings["learning_rate"], weight_decay=settings["weight_decay"])\n    updates = math.ceil(len(loader) / settings["gradient_accumulation"]) * epochs\n    scheduler = trainer.build_scheduler(optimizer, total_updates=updates, warmup_ratio=settings["warmup_ratio"])\n    return optimizer, scheduler, trainer.make_scaler(device)\n\n\ndef _progress_callback(name, every):\n    def report(progress):\n        if progress["successful_updates"] % every == 0:\n            print(f"{name}: {progress[\'examples\']} rows, {progress[\'elapsed_seconds\'] / 60:.1f} minutes", flush=True)\n    return report\n\n\ndef plan_epochs(probes, fit_count, full_count, holdout_count, settings, budget):\n    """Reserve BOTH fits, worst-case full refit, setup, and final export."""\n    batch = settings["batch_size"]\n    accum = settings["gradient_accumulation"]\n    fit_updates = math.ceil(math.ceil(fit_count / batch) / accum)\n    full_updates = math.ceil(math.ceil(full_count / batch) / accum)\n    estimates = {}\n    for name in ORDER:\n        probe = probes[name]\n        step = probe["seconds_per_update"]\n        predict = probe["seconds_per_prediction"]\n        if not all(math.isfinite(v) and v > 0 for v in (step, predict)):\n            raise ValueError("Pilot timing must be finite and positive")\n        estimates[name] = {"fit_epoch_seconds": step * fit_updates,\n            "holdout_seconds": predict * holdout_count, "full_epoch_seconds": step * full_updates}\n    for epochs in range(settings["max_epochs"], 0, -1):\n        fits = sum(row["fit_epoch_seconds"] + row["holdout_seconds"] for row in estimates.values()) * epochs\n        refit = max(row["full_epoch_seconds"] for row in estimates.values()) * epochs\n        # Include repeated model loading/tokenization/checkpoint startup before the export reserve.\n        requested = fits + refit + 240\n        if budget.can_start(requested, training=True):\n            return {"common_epochs": epochs, "estimates": estimates,\n                    "estimated_work_seconds": requested, "remaining_seconds": budget.remaining(),\n                    "safety_factor": budget.safety_factor,\n                    "finalization_reserve_seconds": budget.finalization_reserve_s}\n    raise BudgetExceeded("insufficient time for one matched epoch plus full Training refit")\n\n\ndef choose_candidate(records):\n    control, medical = records["control"], records["medical"]\n    better = (medical["macro_f1"] > control["macro_f1"]\n              and medical["nausea_f1"] >= control["nausea_f1"])\n    selected = "medical" if better else "control"\n    return {"selected_candidate": selected, "selected_epochs": records[selected]["selected_epochs"],\n        "medical_improved": better,\n        "rule": "medical macro F1 > control and medical nausea F1 >= control; otherwise control",\n        "selection_data": "Training internal holdout only",\n        "macro_f1_delta": medical["macro_f1"] - control["macro_f1"],\n        "nausea_f1_delta": medical["nausea_f1"] - control["nausea_f1"],\n        "historical_best_comparison": "not evaluated here; control is a fresh single KLUE model, not the eleven-model ensemble"}\n\n\ndef _probe(spec, texts, targets, fit, holdout, settings, device, budget):\n    budget.check("load pilot", training=True)\n    model, tokenizer = _create(spec, settings, device)\n    loader = _loader(tokenizer, texts, targets, fit, settings, shuffle=True, device=device)\n    validation = _loader(tokenizer, texts, None, holdout, settings, device=device)\n    optimizer, scheduler, scaler = _optimizer(model, loader, settings["max_epochs"], settings, device)\n    stats = trainer.train_epoch(model, loader, optimizer, device=device,\n        pos_weight=trainer.positive_weights(targets[fit]), accumulation_steps=settings["gradient_accumulation"],\n        scheduler=scheduler, scaler=scaler, budget=budget, max_updates=settings["pilot_optimizer_steps"] + 1)\n    measured = stats["measured_update_seconds"]\n    if len(measured) < settings["pilot_optimizer_steps"]:\n        raise ValueError("Pilot requires at least eleven successful optimizer updates")\n    prediction = trainer.predict_logits(model, validation, device=device, budget=_TrainingBudget(budget),\n        max_batches=32, return_stats=True)\n    result = {"successful_updates": stats["successful_updates"], "measured_updates": len(measured),\n        "seconds_per_update": float(max(np.mean(measured), np.percentile(measured, 75))),\n        "seconds_per_prediction": float(prediction["seconds_per_example"]),\n        "probe_examples": stats["examples"], "prediction_examples": prediction["examples"],\n        "pilot_weights_discarded": True}\n    del model, tokenizer, loader, validation, optimizer, scheduler, scaler\n    _release()\n    return result\n\n\ndef _fit_candidate(name, spec, texts, targets, fit, holdout, settings, plan, device, budget, root):\n    model, tokenizer = _create(spec, settings, device)\n    loader = _loader(tokenizer, texts, targets, fit, settings, shuffle=True, device=device)\n    validation = _loader(tokenizer, texts, None, holdout, settings, device=device)\n    optimizer, scheduler, scaler = _optimizer(model, loader, plan["common_epochs"], settings, device)\n    weights = trainer.positive_weights(targets[fit])\n    best, best_logits, history = None, None, []\n    for epoch in range(1, plan["common_epochs"] + 1):\n        print(f"{name}: internal fit epoch {epoch}/{plan[\'common_epochs\']}", flush=True)\n        stats = trainer.train_epoch(model, loader, optimizer, device=device, pos_weight=weights,\n            accumulation_steps=settings["gradient_accumulation"], scheduler=scheduler, scaler=scaler, budget=budget,\n            on_update=_progress_callback(name, settings["log_every"]))\n        if not stats["completed_epoch"]:\n            raise RuntimeError("Candidate epoch did not finish")\n        logits = trainer.predict_logits(model, validation, device=device, budget=_TrainingBudget(budget))\n        metrics = data.score_predictions(targets[holdout], trainer.fixed_decisions(logits))\n        record = {"candidate": name, "macro_f1": metrics["macro_f1"],\n            "nausea_f1": metrics["per_symptom"]["오심"]["f1"], "selected_epochs": epoch,\n            "train_seconds": stats["elapsed_seconds"], "stop_reason": "completed_matched_epochs"}\n        history.append({**record, "metrics": metrics})\n        if best is None or record["macro_f1"] > best["macro_f1"]:\n            best, best_logits = record, logits.copy()\n            data.write_json(root / (name + "_training_metrics.json"), metrics)\n            _save_npz(root / (name + "_holdout_predictions.npz"), indices=holdout, targets=targets[holdout],\n                      logits=best_logits, predictions=trainer.fixed_decisions(best_logits))\n        data.write_json(root / (name + "_history.json"), history)\n        print(f"{name}: Macro F1={record[\'macro_f1\']:.6f}, nausea F1={record[\'nausea_f1\']:.6f}", flush=True)\n    del model, tokenizer, loader, validation, optimizer, scheduler, scaler\n    _release()\n    best["positive_weights"] = weights.tolist()\n    best["actual_epoch_seconds"] = max(row["train_seconds"] for row in history)\n    return best, best_logits\n\n\ndef _save_comparison(root, records, logits, targets, holdout):\n    columns = ["candidate", "macro_f1", "nausea_f1", "selected_epochs", "stop_reason"]\n    with (root / "candidate_summary.csv").open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns)\n        writer.writeheader()\n        writer.writerows({key: records[name][key] for key in columns} for name in ORDER)\n    before, after = [trainer.fixed_decisions(logits[name]) for name in ("control", "medical")]\n    per_label = {}\n    truth = targets[holdout]\n    for j, name in enumerate(data.TARGET_SYMPTOMS):\n        per_label[name] = {"fn_to_tp": int(((truth[:, j] == 1) & (before[:, j] == 0) & (after[:, j] == 1)).sum()),\n            "tp_to_fn": int(((truth[:, j] == 1) & (before[:, j] == 1) & (after[:, j] == 0)).sum()),\n            "fp_to_tn": int(((truth[:, j] == 0) & (before[:, j] == 1) & (after[:, j] == 0)).sum()),\n            "tn_to_fp": int(((truth[:, j] == 0) & (before[:, j] == 0) & (after[:, j] == 1)).sum())}\n    data.write_json(root / "paired_error_changes.json", {"direction": "control to medical", "per_symptom": per_label})\n    probabilities = {name: 1 / (1 + np.exp(-np.clip(logits[name][:, NAUSEA], -80, 80))) for name in ORDER}\n    disagreement = np.minimum(abs(probabilities["medical"] - truth[:, NAUSEA]),\n                              abs(probabilities["control"] - truth[:, NAUSEA]))\n    # Models never saw these holdout rows during fitting. Ranking is for review only.\n    indices = np.argsort(-disagreement, kind="stable")[:200]\n    with (root / "training_review_candidates.csv").open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["training_row_index", "original_nausea_label", "medical_probability", "control_probability",\n                         "both_disagree_with_original", "review_priority"])\n        for i in indices:\n            writer.writerow([int(holdout[i]), int(truth[i, NAUSEA]), float(probabilities["medical"][i]),\n                float(probabilities["control"][i]), bool(before[i, NAUSEA] != truth[i, NAUSEA]\n                and after[i, NAUSEA] != truth[i, NAUSEA]), float(disagreement[i])])\n    data.write_json(root / "review_note.json", {"source": "Training internal holdout; zero-based sorted Training JSON row index",\n        "labels_changed": 0, "samples_removed": 0, "label_errors_confirmed": 0,\n        "warning": "Agreement against a provided label is not proof of label error. Review dialogue evidence manually. Do not use this file as a training mask."})\n\n\ndef _refit(spec, selection, record, texts, targets, settings, plan, device, budget, root):\n    epochs = selection["selected_epochs"]\n    estimate = max(plan["estimates"][selection["selected_candidate"]]["full_epoch_seconds"],\n                   record["actual_epoch_seconds"] * len(texts) / record["fitted_samples"]) * epochs + 60\n    if not budget.can_start(estimate, training=True):\n        raise BudgetExceeded("insufficient remaining time for full Training refit")\n    model, tokenizer = _create(spec, settings, device)\n    loader = _loader(tokenizer, texts, targets, None, settings, shuffle=True, device=device)\n    # Preserve the selected fit\'s relative LR schedule; stopping epoch may be earlier than the horizon.\n    optimizer, scheduler, scaler = _optimizer(model, loader, plan["common_epochs"], settings, device)\n    weights = trainer.positive_weights(targets)\n    history = []\n    for epoch in range(1, epochs + 1):\n        print(f"Full Training refit epoch {epoch}/{epochs}", flush=True)\n        stats = trainer.train_epoch(model, loader, optimizer, device=device, pos_weight=weights,\n            accumulation_steps=settings["gradient_accumulation"], scheduler=scheduler, scaler=scaler, budget=budget,\n            on_update=_progress_callback("full refit", settings["log_every"]))\n        if not stats["completed_epoch"]:\n            raise RuntimeError("Full Training epoch did not finish")\n        history.append({"epoch": epoch, "examples": stats["examples"], "seconds": stats["elapsed_seconds"]})\n        data.write_json(root / "final_history.json", history)\n    metadata = {**selection, "version": "ver24", "model_spec": spec, "training_scope": "all_training",\n        "fitted_samples": len(texts), "scheduler_horizon_epochs": plan["common_epochs"],\n        "positive_weights": weights.tolist(), "max_length": settings["max_length"],\n        "batch_size": settings["batch_size"], "seed": settings["seed"],\n        "decision": "fixed binary logits >= 0; no threshold search, calibration, or forced positives",\n        "input": "utterances[].text only; NFC whitespace normalization; first 512 tokenizer tokens",\n        "target_symptoms": list(data.TARGET_SYMPTOMS), "training_dataset_digest": dataset_digest(texts, targets)}\n    budget.check("export final model")\n    trainer.export_model(root / "final_model", model, tokenizer, metadata)\n    # Compare the exact emitted checkpoint against the in-memory model, then use only the reload for Validation.\n    sample_texts = [texts[index] for index in np.linspace(0, len(texts) - 1, min(16, len(texts)), dtype=int)]\n    sample_loader = _loader(tokenizer, sample_texts, None, None, settings, device=device)\n    expected = trainer.predict_logits(model, sample_loader, device=device, budget=budget)\n    del model, tokenizer, loader, sample_loader, optimizer, scheduler, scaler\n    _release()\n    loaded, tokenizer, saved = trainer.load_export(root / "final_model", device=device)\n    loaded_loader = _loader(tokenizer, sample_texts, None, None, settings, device=device)\n    actual = trainer.predict_logits(loaded, loaded_loader, device=device, budget=budget)\n    if (not np.allclose(expected, actual, rtol=1e-4, atol=1e-4)\n            or not np.array_equal(trainer.fixed_decisions(expected), trainer.fixed_decisions(actual))):\n        raise ValueError("Exported model does not reproduce the saved model predictions")\n    data.write_json(root / "portable_verification.json", {"samples": len(sample_texts), "decisions_equal": True,\n        "max_abs_logit_difference": float(abs(expected - actual).max()), "local_files_only": True})\n    return loaded, tokenizer, metadata\n\n\ndef run(settings, runtime_sources=None):\n    settings = validate_settings(settings)\n    started = time.monotonic()\n    deadline = min(float(settings.get("deadline_monotonic", started + 180 * 60)),\n                   started + settings["time_budget_minutes"] * 60)\n    budget = Budget(deadline, settings["finalization_reserve_seconds"], settings["budget_safety_factor"])\n    device = _device(settings)\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    root = Path(settings["output_dir"]).expanduser().resolve() / stamp\n    root.mkdir(parents=True, exist_ok=False)\n    stage = "prepare_data"\n    data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n    data.write_json(root / "settings.json", settings)\n    try:\n        budget.check(stage, training=True)\n        train_dir, validation_dir = prepare_data(settings["input_zip"], settings["work_dir"],\n            train_dir=settings["train_dir"], validation_dir=settings["validation_dir"])\n        _, texts, targets = data.load_dataset(train_dir, include_targets=True)\n        fit, holdout = data.make_split(texts, targets, settings["holdout_size"], settings["seed"])\n        _save_npz(root / "split.npz", fit=fit, holdout=holdout)\n        data.write_json(root / "training_data.json", {"samples": len(texts), "fit_samples": len(fit),\n            "holdout_samples": len(holdout), "digest": dataset_digest(texts, targets),\n            "labels_changed": 0, "samples_removed": 0})\n        data.write_json(root / "training_diagnostics.json", data.training_diagnostics(texts, targets))\n        specs = read_json(Path(__file__).with_name("model_specs.json"))\n        data.write_json(root / "model_specs.json", specs)\n        probes = {}\n        for name in ORDER:\n            stage = name + "_pilot"\n            data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n            print(f"Measuring {name} GPU throughput; pilot weights will be discarded.", flush=True)\n            probes[name] = _probe(specs[name], texts, targets, fit, holdout, settings, device, budget)\n            data.write_json(root / "pilots.json", probes)\n        plan = plan_epochs(probes, len(fit), len(texts), len(holdout), settings, budget)\n        data.write_json(root / "budget_plan.json", plan)\n        print(f"Budget permits {plan[\'common_epochs\']} matched epoch(s) and a fresh full Training refit.", flush=True)\n        records, logits = {}, {}\n        for name in ORDER:\n            stage = name + "_fit"\n            data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n            records[name], logits[name] = _fit_candidate(name, specs[name], texts, targets, fit, holdout,\n                settings, plan, device, budget, root)\n            records[name]["fitted_samples"] = len(fit)\n        stage = "selection"\n        _save_comparison(root, records, logits, targets, holdout)\n        selection = choose_candidate(records)\n        selection["common_epochs"] = plan["common_epochs"]\n        data.write_json(root / "final_selection.json", selection)\n        selected = selection["selected_candidate"]\n        data.write_json(root / "training_metrics.json", data.score_predictions(targets[holdout],\n            trainer.fixed_decisions(logits[selected])))\n        _save_npz(root / "holdout_predictions.npz", indices=holdout, targets=targets[holdout],\n            logits=logits[selected], predictions=trainer.fixed_decisions(logits[selected]))\n        stage = "full_training_refit"\n        data.write_json(root / "runtime.json", {"status": "running", "stage": stage})\n        model, tokenizer, metadata = _refit(specs[selected], selection, records[selected], texts, targets,\n            settings, plan, device, budget, root)\n        data.write_json(root / "mission3_ver24_model.ckpt", {"format": "mission3_ver24_pointer_v1",\n            "model_directory": "final_model", **metadata})\n        stage = "final_validation"\n        if settings["run_validation"]:\n            # Official Validation is opened only after selection and complete full Training refit.\n            budget.check(stage)\n            paths, validation_texts, validation_targets = data.load_dataset(validation_dir, include_targets=True)\n            loader = _loader(tokenizer, validation_texts, None, None, settings, device=device)\n            final_logits = trainer.predict_logits(model, loader, device=device, budget=budget)\n            predictions = trainer.fixed_decisions(final_logits)\n            data.write_predictions(paths, predictions, root / "mission3.csv")\n            data.write_json(root / "validation_metrics.json", data.score_predictions(validation_targets, predictions))\n            _save_npz(root / "validation_predictions.npz", logits=final_logits,\n                targets=validation_targets, predictions=predictions)\n            del loader\n        del model, tokenizer\n        _release()\n        stage = "finalize"\n        sources = runtime_sources or {path.name: path.read_text(encoding="utf-8") for path in Path(__file__).parent.iterdir()\n            if path.suffix in {".py", ".txt", ".json", ".md"} and not path.name.startswith("test_")}\n        submission = root / "submission"\n        submission.mkdir()\n        for name, content in sources.items():\n            if Path(name).name != name:\n                raise ValueError("Unsafe runtime source name")\n            budget.check(stage)\n            (submission / name).write_text(content, encoding="utf-8")\n        data.write_json(root / "runtime.json", {"status": "complete", "stage": "complete", "device": device,\n            "elapsed_worker_seconds": time.monotonic() - started, "medical_improved": selection["medical_improved"]})\n        hashes = {path.relative_to(root).as_posix(): _hash(path, budget)\n                  for path in sorted(root.rglob("*")) if path.is_file()}\n        budget.check("publish completion marker")\n        data.write_json(root / "run_complete.json", {"version": "ver24", "status": "complete",\n            "hashes": hashes, "training_scope": "all_training", "fitted_samples": len(texts),\n            "validation_evaluated": settings["run_validation"]})\n        return root\n    except Exception as error:\n        data.write_json(root / "runtime.json", {"status": "incomplete", "stage": stage,\n            "error_type": type(error).__name__, "elapsed_worker_seconds": time.monotonic() - started})\n        raise\n\n\ndef _model_directory(checkpoint):\n    checkpoint = Path(checkpoint).resolve()\n    if checkpoint.is_dir():\n        return checkpoint\n    manifest = read_json(checkpoint)\n    if manifest.get("format") != "mission3_ver24_pointer_v1" or manifest.get("model_directory") != "final_model":\n        raise ValueError("Unrecognized ver24 model pointer")\n    result = checkpoint.parent / "final_model"\n    if result.is_symlink() or not result.resolve().is_relative_to(checkpoint.parent):\n        raise ValueError("Model directory escapes the result folder")\n    return result\n\n\ndef inference(checkpoint, label_dir, output):\n    import torch\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    model, tokenizer, metadata = trainer.load_export(_model_directory(checkpoint), device=device)\n    paths, texts, _ = data.load_dataset(label_dir, include_targets=False)\n    loader = trainer.make_loader(tokenizer, texts, batch_size=metadata["batch_size"], max_length=metadata["max_length"])\n    logits = trainer.predict_logits(model, loader, device=device)\n    data.write_predictions(paths, trainer.fixed_decisions(logits), output)\n    return Path(output)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--checkpoint", required=True)\n    parser.add_argument("--label_dir", required=True)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args(argv)\n    inference(args.checkpoint, args.label_dir, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'inference.py': '"""One-command inference entry point for a portable ver24 result folder."""\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nif __package__:\n    from .runtime import main\nelse:\n    root = Path(__file__).resolve().parent\n    name = "mission3_ver24_submission"\n    spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                  submodule_search_locations=[str(root)])\n    package = importlib.util.module_from_spec(spec)\n    sys.modules[name] = package\n    spec.loader.exec_module(package)\n    from importlib import import_module\n    main = import_module(name + ".runtime").main\n\nif __name__ == "__main__":\n    main()\n', 'run_notebook.py': '"""Launch the bounded encoder comparison in a fresh interpreter."""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport re\nimport sys\nimport traceback\n\nSOURCE_FILES = (\n    "__init__.py", "data.py", "bootstrap.py", "budget.py", "trainer.py", "runtime.py",\n    "inference.py", "run_notebook.py", "requirements.txt", "README.md", "model_specs.json",\n)\n\n\ndef _failure_report(error, stage):\n    """Report diagnostic categories without copying raw text or exception values."""\n    category = "unclassified"\n    reason = "실행 중 오류가 발생했습니다. 실패 단계와 코드 위치를 포함한 오류 보고서를 확인하세요."\n    try:\n        message = str(error)\n    except Exception:\n        message = ""\n    if stage == "settings_load":\n        category = "settings_unreadable"\n        reason = "실행 설정을 읽지 못했습니다. 수정된 노트북을 처음 셀부터 모두 실행하세요."\n    elif isinstance(error, ImportError):\n        category = "dependency_import_failed"\n        reason = "필요한 패키지를 불러오지 못했습니다. 패키지 준비 셀의 설치 결과를 확인하세요."\n    elif ("Colab에서 GPU 런타임을 선택하세요" in message\n          or "CUDA GPU" in message or "GPU required" in message):\n        category = "gpu_unavailable"\n        reason = "GPU가 연결되지 않았습니다. Colab 런타임 유형을 GPU로 설정한 뒤 처음부터 실행하세요."\n    elif "CUDA out of memory" in message or type(error).__name__ == "OutOfMemoryError":\n        category = "out_of_memory"\n        reason = "GPU 메모리가 부족합니다. 다른 실행을 종료하고 batch_size를 4로 낮춘 뒤 처음부터 다시 실행하세요. 재실행에는 추가 리소스가 듭니다."\n    elif "Input ZIP is missing:" in message:\n        category = "input_zip_missing"\n        reason = "입력 ZIP을 찾지 못했습니다. Drive 연결과 SETTINGS[\'input_zip\']의 실제 경로를 확인하세요."\n    elif type(error).__name__ == "BudgetExceeded":\n        category = "time_budget_exhausted"\n        reason = "3시간 예산에서 비교와 전체 Training 재학습을 완료하기 어렵거나 시간이 끝나 중단했습니다. 완료 표시가 없는 결과는 제출에 사용하지 마세요."\n    elif any(token in message for token in (\n        "huggingface.co", "Connection error", "Can\'t load tokenizer", "Can\'t load the model",\n        "GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError",\n    )) or type(error).__name__ in {"GatedRepoError", "RepositoryNotFoundError", "RevisionNotFoundError"}:\n        category = "pretrained_model_unavailable"\n        reason = "고정된 공개 모델 또는 토크나이저를 내려받지 못했습니다. Colab 인터넷 연결과 model_specs.json의 모델 공개 상태를 확인하세요."\n    elif stage == "source_read":\n        category = "runtime_sources_unreadable"\n        reason = "노트북에 포함된 실행 코드 일부를 읽지 못했습니다. 수정된 노트북을 처음 셀부터 모두 실행하세요."\n    elif stage == "result_write":\n        category = "result_record_failed"\n        reason = "실행 결과 위치를 기록하지 못했습니다. Drive 결과 폴더의 완료 표시와 오류 보고서를 확인하세요."\n    frames = [{"file": Path(frame.f_code.co_filename).name, "line": line,\n               "function": frame.f_code.co_name}\n              for frame, line in traceback.walk_tb(error.__traceback__)]\n    report = {"status": "failed", "stage": stage, "error_type": type(error).__name__,\n              "category": category, "reason": reason, "frames": frames[-20:]}\n    if (isinstance(error, ImportError) and isinstance(error.name, str)\n            and re.fullmatch(r"[A-Za-z_]\\w*(?:\\.[A-Za-z_]\\w*)*", error.name, flags=re.ASCII)):\n        report["module_name"] = error.name\n    if isinstance(error, OSError) and type(error.errno) is int:\n        report["errno"] = error.errno\n    return report\n\n\ndef _write_report(path, report):\n    path.write_text(json.dumps(report, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    parser.add_argument("--result_file", required=True)\n    args = parser.parse_args(argv)\n    root = Path(__file__).resolve().parent\n    error_file = root / "notebook_error.json"\n    stage = "initialize"\n    try:\n        _write_report(error_file, {"status": "starting", "stage": stage})\n        stage = "settings_load"\n        settings = json.loads(Path(args.settings).read_text(encoding="utf-8-sig"))\n        if not isinstance(settings, dict):\n            raise ValueError("Notebook settings must be a JSON object")\n        stage = "runtime_import"\n        name = "mission3_ver24_execution"\n        spec = importlib.util.spec_from_file_location(name, root / "__init__.py",\n                                                      submodule_search_locations=[str(root)])\n        package = importlib.util.module_from_spec(spec)\n        sys.modules[name] = package\n        spec.loader.exec_module(package)\n        runtime = importlib.import_module(name + ".runtime")\n        stage = "source_read"\n        sources = {name: (root / name).read_text(encoding="utf-8") for name in SOURCE_FILES}\n        stage = "runtime_run"\n        result = runtime.run(settings, runtime_sources=sources)\n        stage = "result_write"\n        Path(args.result_file).write_text(json.dumps(str(result)), encoding="utf-8")\n        _write_report(error_file, {"status": "complete", "stage": "complete"})\n    except Exception as error:\n        report = _failure_report(error, stage)\n        try:\n            _write_report(error_file, report)\n        except OSError:\n            report["diagnostic_file_saved"] = False\n        print(json.dumps(report, ensure_ascii=False, allow_nan=False), flush=True)\n        raise SystemExit(1) from None\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'torch>=2.6,<3\ntransformers==4.57.1\nnumpy==2.1.3\nscikit-learn==1.6.1\nscipy>=1.11,<2\niterative-stratification==0.1.9\njoblib>=1.3,<2\nsentencepiece>=0.2,<0.3\nprotobuf>=4.25,<7\n', 'README.md': '# Mission 3 ver24 — 의료 사전학습 encoder 비교\n\n`mission3_ver24.ipynb`를 Colab GPU 런타임에서 **모두 실행**합니다. Drive 연결에는 기존 입력 ZIP이 있는 공용 계정을 선택합니다. 추가 업로드는 기존 `mission3_ver5_inputs.zip`만 필요합니다. 이전 버전의 모델 파일은 필요하지 않습니다. 모델 다운로드에는 인터넷 연결이 필요합니다.\n\n## 이번 실험의 목적\n\n의료 문장으로 사전학습된 `silber2258/bi-KM-BERT`가 동일한 분류 학습을 거친 `klue/roberta-base`보다 전체 증상과 오심을 잘 구분하는지 확인합니다. 두 모델의 공개 revision은 `model_specs.json`에 고정합니다. bi-KM-BERT는 한국어·영어 의료 corpus로 추가 사전학습한 BERT이며, 이 과제의 9개 증상을 이미 학습한 분류기는 아닙니다.\n\n이 실험은 이전 ver19~23 모델의 추가 학습이나 친구의 11개 모델 앙상블 재현이 아닙니다. 의료 사전학습의 효과를 비교하는 새 실험입니다. 기존 최고 모델보다 성능이 높다는 가정은 하지 않습니다.\n\n## 데이터와 비교 조건\n\n- 원본 서울 Training 데이터와 9개 정답을 유지합니다. 의심 정답 자동 삭제, 재라벨링, label masking, 외부 학습 데이터 추가를 하지 않습니다.\n- 입력은 `utterances[].text`이며 기존 데이터의 `utterences` 표기도 지원합니다. 다른 메타데이터를 입력으로 사용하지 않습니다.\n- 같은 원문 대화가 fit과 holdout 양쪽에 들어가지 않는 Training 내부 분할을 seed 42로 만듭니다. holdout 비율은 약 20%입니다. 동일 대화 그룹 보존 때문에 정확한 행 수는 데이터에 따라 달라질 수 있습니다.\n- 두 후보의 학습 데이터, 평가 데이터, 학습률, 손실, batch 설정 및 최대 epoch를 맞춥니다. 모델별로 해당 사전학습 tokenizer를 사용합니다. tokenizer의 차이 때문에 같은 512토큰이 포함하는 원문 길이는 다를 수 있습니다.\n- 각 대화의 앞부분을 최대 512토큰으로 사용합니다. 모든 발화를 별도 창으로 보존한 ver19와 입력 처리 방식이 다르므로, 내부 점수를 ver19와 동일 조건 비교로 해석하지 않습니다.\n- mean pooling, dropout 0.1, 9개 출력으로 증상을 분류합니다. 원문이나 환자별 진단 정보를 진행 로그에 출력하지 않습니다.\n\n손실은 `BCEWithLogitsLoss(pos_weight=...)`입니다. 각 증상의 양성 가중치는 학습에 쓰는 데이터에서 계산한 `clip(sqrt(negative_count / positive_count), 1, 5)`를 사용합니다. 내부 비교에서는 fit 데이터에서만 계산하며, 최종 재학습에서는 전체 Training에서 다시 계산합니다. 양성·음성 수가 0인 경우는 구현의 수치 안전 처리를 따릅니다. 검증 데이터로 가중치를 맞추지 않습니다.\n\n모든 증상의 최종 예측은 **`logit >= 0`**으로 고정합니다. 이는 sigmoid 확률 0.5와 같습니다. threshold 탐색, 증상별 기준 변경, 사후 calibration, 확률 이동, 앙상블 비율 탐색을 적용하지 않습니다.\n\n## 학습 순서\n\n1. Drive 연결 후 180분 공유 예산을 시작하고 기존 버전과 같은 패키지를 설치합니다. 별도 Python 프로세스에서 실행하므로 이미 로드된 Colab 패키지와 새 설치가 섞이지 않도록 합니다.\n2. 입력 ZIP과 공개 모델을 준비하고 두 모델의 짧은 실제 학습 속도를 측정합니다. 속도 측정용 학습은 후보의 최종 시작 가중치로 재사용하지 않습니다.\n3. 두 후보의 학습·내부 평가와 선택 모델의 전체 Training 재학습까지 고려해 가능한 **공통 최대 epoch 수**를 최대 3 안에서 정합니다. 느린 장치에서는 더 적은 epoch로 진행하거나 명확히 중단합니다.\n4. 두 후보를 사전학습 가중치에서 각각 새로 학습합니다. 각 후보의 내부 평가 Macro F1이 가장 좋은 epoch 수·예측·지표를 저장합니다. Drive 저장량을 줄이기 위해 비교용 fit 가중치는 보관하지 않으며, 최종 전체 Training 재학습 가중치만 보관합니다.\n5. **의료 후보의 Macro F1이 대조군보다 높고 오심 F1도 낮아지지 않을 때만 의료 후보를 선택**합니다. 그렇지 않으면 이번 실행의 일반 한국어 대조군을 선택합니다.\n6. 선택한 모델 종류와 epoch 수로 전체 Training을 사전학습 가중치에서 다시 학습합니다. fit-only 모델을 최종 제출 모델로 표시하지 않습니다.\n7. 저장한 모델을 다시 불러와 추론이 일치하는지 검증한 뒤 공식 Validation을 평가하고 완료 표시를 남깁니다. Validation은 후보·epoch 선택에 쓰지 않습니다.\n\n기본 설정은 batch 8, gradient accumulation 2, 학습률 2e-5, weight decay 0.01, warmup 비율 0.1입니다. 학습 속도 추정에는 안전 계수 1.35를 적용하며 최종 평가·저장용으로 20분을 예약합니다. 180분은 설치·다운로드·준비·학습·저장을 합한 실행 예산이며, 모든 GPU에서 모든 단계의 완료를 보장하는 시간이 아닙니다. 부족한 예산에서 완료 결과를 가장하지 않습니다.\n\n학습률 스케줄의 기준 기간은 내부 fit과 전체 Training 재학습 모두 속도 측정 후 정한 `common_epochs`입니다. 선택된 epoch가 이보다 적더라도 스케줄을 그 짧은 기간으로 다시 압축하지 않습니다. 각 단계의 전체 데이터 수에 맞춰 step 수를 계산하므로, 선택 epoch까지의 상대적 학습률 변화를 유지합니다.\n\n## 결과 파일\n\n기본 결과 위치는 `MyDrive/DCC_AIdeas/results/mission3/ver24/<run_id>/`입니다.\n\n| 파일 | 용도 |\n|---|---|\n| `candidate_summary.csv` | 같은 holdout에서 후보별 Macro F1, 오심 F1, 선택 epoch 비교 |\n| `final_selection.json` | 선택 후보와 이유 |\n| `training_metrics.json` | 선택한 fit 후보의 내부 holdout 점수 |\n| `paired_error_changes.json` | 일반 한국어 모델 대비 의료 모델의 누락 회복·새 누락·오탐 제거·새 오탐 |\n| `training_review_candidates.csv`, `review_note.json` | 원래 오심 정답과 두 모델 판단이 많이 다른 Training holdout 검토 후보 최대 200건 |\n| `validation_metrics.json` | 전체 Training 재학습 모델의 공식 Validation 점수 |\n| `mission3.csv` | 최종 Validation 예측 |\n| `mission3_ver24_model.ckpt`, `final_model/` | 최종 모델 manifest와 가중치·tokenizer 등 |\n| `runtime.json`, `run_complete.json` | 실행 상태와 완료 표시 |\n\n`run_complete.json`과 최종 CSV가 없으면 완료된 제출 결과로 사용하지 않습니다. `run_validation=False`는 모델 검증용으로 CSV를 만들지 않으므로 기본값 True를 유지합니다. manifest만 단독 보관하지 말고 **실행 폴더 전체를 보관**합니다.\n\n검토 후보 파일에는 정렬된 Training 데이터의 0부터 시작하는 행 번호, 원래 정답, 두 모델의 오심 확률을 저장합니다. 대화 원문과 환자 파일 이름은 포함하지 않습니다. 모델 둘이 원래 정답과 다르게 판단하더라도 정답 오류가 확정된 것은 아닙니다. 원문을 읽어 근거를 검토하기 위한 목록이며, 이 버전은 목록을 정답 수정이나 학습 제외에 사용하지 않습니다.\n\n실패 시 노트북은 GPU 반환 전에 실패 단계·오류 종류·코드 위치를 표시하고 `ver24/_diagnostics/ver24_error_*.json`에 저장합니다. 오류 보고서에는 원문 대화와 개별 환자 파일 이름을 포함하지 않습니다. 학습 중단 후 자동 재개는 지원하지 않습니다. 다시 실행하면 새 예산과 Colab 리소스를 사용합니다. 자동 GPU 반환이 실패하면 Colab에서 직접 연결 해제 및 런타임 삭제를 실행합니다.\n\n## 검증 범위\n\n로컬에서 코드와 노트북 생성, 실행 준비·오류 처리, 데이터 분리·예산·저장 관련 검증을 수행할 수 있습니다. 실제 F1, GPU 학습 속도, Drive 속도, Colab 자동 런타임 반환은 실제 Colab 실행에서 확인해야 합니다. 원문에 증상 단어가 없다는 이유만으로 정답 오류가 확정되는 것은 아니므로, 의심 라벨은 이 버전에서 학습 대상에서 제외하지 않습니다.\n\n공개 모델 출처: [KU-RIAS bi-medical-bert](https://github.com/KU-RIAS/bi-medical-bert), [bi-KM-BERT](https://huggingface.co/silber2258/bi-KM-BERT), [KLUE-RoBERTa-base](https://huggingface.co/klue/roberta-base). 체크포인트별 상세 출처와 revision은 `model_specs.json`을 확인합니다.\n\n검토한 bi-KM-BERT 모델 카드에는 가중치의 명시적 라이선스가 없었습니다. 공개 다운로드 가능 여부와 재배포 허용 여부는 다르며, 이 정보는 `model_specs.json`에도 기록합니다. 해당 모델의 라이선스 조건이 확인됐다고 주장하지 않습니다.\n\n저장된 결과 폴더의 `submission/inference.py`는 `--checkpoint <run>/mission3_ver24_model.ckpt --label_dir <label_directory> --output <output_csv>` 인자로 추론합니다. 실행 위치가 달라도 manifest와 `final_model/`의 상대 위치를 유지해야 합니다.\n', 'model_specs.json': '{\n  "medical": {\n    "model_id": "silber2258/bi-KM-BERT",\n    "revision": "f7f66bb5b1c1a3e7b71afc9deefe80e7c123c5e5",\n    "source": "https://github.com/KU-RIAS/bi-medical-bert",\n    "model_card": "https://huggingface.co/silber2258/bi-KM-BERT",\n    "license_note": "No explicit weight license in the inspected model card; public access is not a redistribution license."\n  },\n  "control": {\n    "model_id": "klue/roberta-base",\n    "revision": "02f94ba5e3fcb7e2a58a390b8639b0fac974a8da",\n    "source": "https://huggingface.co/klue/roberta-base",\n    "license_note": "CC-BY-SA-4.0 per upstream model card."\n  }\n}\n'}
RUNTIME_PREPARED = None
DEPENDENCIES_PREPARED = None
if ("DEADLINE_MONOTONIC" not in globals() or "time" not in globals()
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("작업 예산 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
try:
    if time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    runtime_dir = Path(tempfile.mkdtemp(prefix="mission3_ver24_"))
    for name, source in RUNTIME_SOURCES.items():
        (runtime_dir / name).write_bytes(source.encode("utf-8"))
    budget_spec = importlib.util.spec_from_file_location("mission3_ver24_budget_supervisor", runtime_dir / "budget.py")
    budget_module = importlib.util.module_from_spec(budget_spec)
    sys.modules[budget_spec.name] = budget_module
    budget_spec.loader.exec_module(budget_module)
    RUNTIME_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)
    print("실행 코드 준비 완료")
except BaseException:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")
    raise


In [ ]:
# @title 기존 프로젝트와 같은 버전의 패키지 준비 (최대 20분, 전체 예산에 포함)
DEPENDENCIES_PREPARED = None
required = ("runtime_dir", "budget_module", "DEADLINE_MONOTONIC", "time", "sys")
if (any(name not in globals() for name in required)
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("실행 코드 준비가 완료되지 않았습니다. 런타임 → 모두 실행으로 처음 셀부터 실행하세요.")
if (globals().get("RUNTIME_PREPARED") != (str(runtime_dir), DEADLINE_MONOTONIC)
        or not (runtime_dir / "requirements.txt").is_file()):
    raise RuntimeError("현재 실행의 코드 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")
try:
    if time.monotonic() >= DEADLINE_MONOTONIC:
        raise RuntimeError("이번 작업의 시간 예산이 끝났습니다. 다시 실행하려면 처음 셀부터 모두 실행하세요.")
    package_deadline = min(DEADLINE_MONOTONIC, time.monotonic() + 20 * 60)
    budget_module.run_supervised(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(runtime_dir / "requirements.txt")],
        deadline_monotonic=package_deadline, shutdown_margin_s=10,
    )
    DEPENDENCIES_PREPARED = (str(runtime_dir), DEADLINE_MONOTONIC)
    print("패키지 준비 완료 · 남은 작업 시간:", round(max(0, DEADLINE_MONOTONIC - time.monotonic()) / 60, 1), "분")
except BaseException:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· 위의 최초 오류를 확인하세요.")
    raise


In [ ]:
# @title 속도 측정 → 두 모델 비교 → 전체 Training 재학습 → 최종 평가 · GPU 반환
import csv
import json
import subprocess
required = ("runtime_dir", "SETTINGS", "budget_module", "DEADLINE_MONOTONIC",
            "BUDGET_STARTED_MONOTONIC", "time", "Path", "sys")
if (any(name not in globals() for name in required)
        or not callable(globals().get("release_gpu"))):
    raise RuntimeError("준비 셀이 완료되지 않았거나 런타임이 초기화됐습니다. 마지막 셀만 실행하지 말고 런타임 → 모두 실행으로 처음부터 실행하세요. 앞선 셀이 실패했다면 그 셀의 첫 오류를 확인하세요.")
prepared_context = (str(runtime_dir), DEADLINE_MONOTONIC)
if (globals().get("RUNTIME_PREPARED") != prepared_context
        or globals().get("DEPENDENCIES_PREPARED") != prepared_context
        or not (runtime_dir / "run_notebook.py").is_file()):
    raise RuntimeError("현재 실행의 코드 또는 패키지 준비가 미완료입니다. 앞선 셀의 첫 오류를 확인하고 처음부터 모두 실행하세요.")
error_file = runtime_dir / "notebook_error.json"
try:
    error_file.write_text(json.dumps({"status": "starting"}), encoding="utf-8")
    if time.monotonic() >= DEADLINE_MONOTONIC:
        raise budget_module.BudgetExceeded("notebook launch", deadline_monotonic=DEADLINE_MONOTONIC)
    settings_file = runtime_dir / "notebook_settings.json"
    result_file = runtime_dir / "notebook_result.json"
    settings_file.write_text(json.dumps(SETTINGS, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    budget_module.run_supervised(
        [sys.executable, "-u", str(runtime_dir / "run_notebook.py"),
         "--settings", str(settings_file), "--result_file", str(result_file)],
        deadline_monotonic=DEADLINE_MONOTONIC, shutdown_margin_s=10,
    )
    result_dir = Path(json.loads(result_file.read_text(encoding="utf-8")))
    print("결과 폴더:", result_dir)
    summary = result_dir / "candidate_summary.csv"
    if summary.is_file():
        print("동일 Training 내부 분할 비교:")
        with summary.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                print(row["candidate"], "Macro F1 =", round(float(row["macro_f1"]), 6),
                      "오심 F1 =", round(float(row["nausea_f1"]), 6),
                      "선택 epoch =", row["selected_epochs"])
    for filename in ["training_metrics.json", "validation_metrics.json"]:
        path = result_dir / filename
        if path.is_file():
            report = json.loads(path.read_text(encoding="utf-8-sig"))
            print(filename, "Macro F1 =", round(report["macro_f1"], 6))
            for symptom, values in report["per_symptom"].items():
                print(symptom, "F1 =", round(values["f1"], 4),
                      "P =", round(values["precision"], 4), "R =", round(values["recall"], 4))
    manifest_path = result_dir / "mission3_ver24_model.ckpt"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        print("저장된 후보:", manifest.get("selected_candidate", "모델 정보 확인"))
        print("저장 모델 학습 범위:", manifest.get("training_scope", "모델 정보 확인"),
              "· 학습 표본:", manifest.get("fitted_samples", "모델 정보 확인"),
              "· 학습 epoch:", manifest.get("selected_epochs", "모델 정보 확인"))
    status_path = result_dir / "runtime.json"
    status = json.loads(status_path.read_text(encoding="utf-8-sig")) if status_path.is_file() else {}
    print("실행 상태:", status.get("status", "기록 확인 필요"))
    print("이번 노트북 소요 시간:", round((time.monotonic() - BUDGET_STARTED_MONOTONIC) / 60, 1), "분")
    print("선택 근거:", result_dir / "final_selection.json")
    complete = ((result_dir / "run_complete.json").is_file() and manifest_path.is_file()
                and (result_dir / "final_model").is_dir()
                and ((result_dir / "mission3.csv").is_file() or not SETTINGS["run_validation"])
                and status.get("status") == "complete")
    if complete:
        if SETTINGS["run_validation"]:
            print("최종 작업 완료 · 전체 Training 재학습과 저장 검증을 마친 예측:", result_dir / "mission3.csv")
        else:
            print("모델 검증 실행 완료 · Validation 예측과 제출 CSV는 만들지 않았습니다.")
            print("제출용 실행은 run_validation=True로 설정해야 합니다.")
        print("이전 최고 모델과의 성능 비교는 validation_metrics.json으로 따로 확인하세요.")
    else:
        print("미완료 실행:", status.get("status", "상태 기록 없음"),
              "· 최종 제출 결과로 사용하지 마세요. 결과 폴더의 로그를 확인하세요.")
except Exception as worker_error:
    failure = {"status": "failed", "stage": "notebook_run", "error_type": type(worker_error).__name__,
               "category": "notebook_execution_failed",
               "reason": "실행 또는 결과 확인 중 오류가 발생했습니다. 실패 단계와 코드 위치를 확인하세요.", "frames": []}
    if isinstance(worker_error, subprocess.CalledProcessError):
        failure.update(stage="worker_process", category="worker_process_failed", exit_code=worker_error.returncode,
                       reason="실행 프로그램이 상세 오류를 저장하기 전에 종료됐습니다. 종료 코드와 준비 상태 확인이 필요합니다.")
        try:
            recorded = json.loads(error_file.read_text(encoding="utf-8"))
            if isinstance(recorded, dict) and recorded.get("status") == "failed":
                failure.update(recorded)
        except (OSError, ValueError):
            pass
    elif type(worker_error).__name__ == "BudgetExceeded":
        failure.update(category="time_budget_exhausted",
                       reason="3시간 작업 예산이 끝나 실행을 중단했습니다. 완료 표시가 없는 결과는 제출에 사용하지 마세요.")
    print("ver24 내부 오류 · 실패 단계:", failure["stage"], "· 오류 종류:", failure["error_type"], flush=True)
    print("확인 내용:", failure["reason"], flush=True)
    if failure.get("module_name"):
        print("불러오지 못한 모듈:", failure["module_name"], flush=True)
    for frame in failure.get("frames", [])[-6:]:
        print("코드 위치:", frame["file"], "·", frame["function"], "·", frame["line"], flush=True)
    saved_error = None
    try:
        from datetime import datetime, timezone
        import uuid
        error_dir = Path(SETTINGS["output_dir"]) / "_diagnostics"
        error_dir.mkdir(parents=True, exist_ok=True)
        saved_error = error_dir / ("ver24_error_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                                  + "_" + uuid.uuid4().hex[:8] + ".json")
        saved_error.write_text(json.dumps(failure, ensure_ascii=False, indent=2), encoding="utf-8")
        print("오류 보고서:", saved_error, flush=True)
    except OSError as save_error:
        saved_error = None
        print("Drive 오류 보고서 저장 실패:", type(save_error).__name__, "· 위의 내부 오류 내용을 복사하세요.", flush=True)
    raise RuntimeError("ver24 실행 실패 [" + failure["stage"] + " / " + failure["error_type"] + "]: "
                       + failure["reason"] + (" 오류 보고서: " + str(saved_error) if saved_error else "")) from None
finally:
    cleanup = globals().get("release_gpu")
    if callable(cleanup):
        try:
            cleanup()
        except Exception as cleanup_error:
            print("GPU 반환 중 오류:", type(cleanup_error).__name__, "· Colab에서 직접 연결을 해제하세요.")


## 결과 확인

결과는 `DCC_AIdeas/results/mission3/ver24/실행시각_고유번호/`에 저장됩니다.
- `candidate_summary.csv`: 같은 내부 holdout에서 두 후보를 비교한 결과
- `final_selection.json`: 모델과 epoch 선택 근거
- `training_metrics.json`: 선택 후보의 Training 내부 holdout 점수
- `paired_error_changes.json`: 의료 모델에서 오심 누락이 회복되거나 새로 생긴 건수 등
- `training_review_candidates.csv`: 원래 오심 정답과 두 모델의 판단이 많이 다른 Training holdout 검토 후보 최대 200건
- `validation_metrics.json`, `mission3.csv`: 전체 Training 재학습 후 공식 Validation 결과
- `runtime.json`, `run_complete.json`: 실행 상태와 완료 표시
- `mission3_ver24_model.ckpt`, `final_model/`: 최종 모델과 재사용에 필요한 파일

**`run_complete.json`과 최종 CSV가 없는 실행은 제출용 완료 결과가 아닙니다.**
모델 manifest만 단독 보관하지 말고 **실행 폴더 전체를 보관**하세요.
검토 후보에는 원문 대신 Training 행 번호와 확률을 저장합니다. 정답 오류 확정 목록이 아니며 학습 제외에 사용하지 않습니다.
실패 시 GPU 반환 전에 실패 단계·오류 종류·코드 위치를 표시하고
`ver24/_diagnostics/ver24_error_*.json`에 같은 오류 보고서를 저장합니다.
이 보고서를 전달하면 원인을 확인할 수 있습니다. 원문 대화나 개별 데이터 파일 이름을 출력하지 않습니다.
중단된 학습의 자동 재개는 지원하지 않습니다. 재실행은 처음부터 시작하며 새 Colab 리소스를 소비합니다.
자동 반환이 실패하면 **런타임 → 연결 해제 및 런타임 삭제**를 직접 실행하세요.
의료 모델의 효과와 GPU 시간, 실제 Colab 자동 반환은 실행 결과로 확인해야 합니다.
bi-KM-BERT 모델 카드에서 가중치의 명시적 라이선스는 확인되지 않았으며, 출처·revision과 함께 `model_specs.json`에 기록합니다.
